In [2]:
# pip installs (uncomment in Colab if needed)
!pip install -q scikit-learn phe tenseal tqdm
!pip install torch_geometric

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.7/53.7 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 41.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 23.6 MB/s eta 0:00:00


# **GIN Model**

In [3]:
# =============================================================================
# Full Cora pipeline with REAL Homomorphic Inference (Paillier + CKKS)
# GIN -> LogisticRegression -> Paillier & CKKS inference
# Aggregated output: mean ± std dictionaries
# =============================================================================

import os, time, random
import numpy as np
import torch
import torch.nn.functional as F
from copy import deepcopy
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, log_loss
from torch_geometric.datasets import Planetoid
from torch_geometric.nn import GINConv
import phe as paillier
import tenseal as ts
from tqdm import tqdm
import pandas as pd

# ---------------- USER KNOBS ----------------
RND = 42
N_FOLDS = 3                # keep small for runtime; increase to 5 if you want
GIN_HIDDEN = 128
GIN_EMB = 64               # embedding dim
GIN_EPOCHS = 15
PAILLIER_SCALE = 5000      # integer quantization scale for Paillier
PAILLIER_KEY = 2048        # bits for paillier keys
CKKS_CTX_SCALE_EXP = 40
CKKS_BATCH_SIZE = 16
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# ---------------------------------------------

random.seed(RND); np.random.seed(RND); torch.manual_seed(RND)
device = torch.device(DEVICE)
print("Device:", device)

# ---------------- Data ----------------
dataset = Planetoid(root="/tmp/Cora_full_he", name="Cora")
data = dataset[0]

# --------------- GIN (node embeddings) ---------------
class GINModel(torch.nn.Module):
    def __init__(self, in_ch, hidden=GIN_HIDDEN, emb_dim=GIN_EMB, dropout=0.3):
        super().__init__()
        def mlp(inp, outp):
            return torch.nn.Sequential(torch.nn.Linear(inp, outp), torch.nn.ReLU(), torch.nn.Linear(outp, outp))
        self.conv1 = GINConv(mlp(in_ch, hidden))
        self.conv2 = GINConv(mlp(hidden, emb_dim))
        self.dropout = torch.nn.Dropout(dropout)

    def forward(self, x, edge_index):
        x = F.relu(self.conv1(x, edge_index))
        x = self.conv2(x, edge_index)
        x = self.dropout(x)
        return x  # per-node embeddings

# train GIN briefly to produce embeddings
model = GINModel(dataset.num_features).to(device)
data = data.to(device)
opt = torch.optim.Adam(model.parameters(), lr=3e-3, weight_decay=5e-5)
crit = torch.nn.CrossEntropyLoss()

model.train()
for ep in range(1, GIN_EPOCHS + 1):
    opt.zero_grad()
    emb = model(data.x, data.edge_index)  # [N, emb_dim]
    # supervised node-level loss (helps representation)
    loss = crit(emb, data.y)
    loss.backward()
    opt.step()
    if ep % 5 == 0:
        print(f"GIN epoch {ep}/{GIN_EPOCHS}, loss={loss.item():.4f}")

model.eval()
with torch.no_grad():
    embeddings = model(data.x, data.edge_index).cpu().numpy()
labels = data.y.cpu().numpy()
print("Embeddings shape:", embeddings.shape)

# scale embeddings before LR
scaler = StandardScaler()
embeddings = scaler.fit_transform(embeddings)

# ---------------- HE helper functions ----------------

# Paillier multiclass inference (exact integer arithmetic)
def paillier_inference_multiclass(X, W, B, scale=PAILLIER_SCALE, key_size=PAILLIER_KEY):
    """
    X: (n, d) numpy
    W: (n_classes, d) numpy
    B: (n_classes,) numpy
    Returns: logits (n, n_classes), wall_time seconds
    """
    pk, sk = paillier.generate_paillier_keypair(n_length=key_size)
    # quantize weights and biases to integers
    W_int = [[int(round(float(wij) * scale)) for wij in wrow] for wrow in W.tolist()]
    B_int = [int(round(float(bi) * (scale ** 2))) for bi in B.tolist()]  # bias scaled by scale^2 (because x* w scaled by scale^2)
    logits = []
    t0 = time.time()
    for x in X:
        # quantize input vector to integers
        x_int = [int(round(float(xj) * scale)) for xj in x.tolist()]
        # encrypt each entry
        enc_x = [pk.encrypt(int(v)) for v in x_int]
        cls_scores = []
        for wrow, b_int in zip(W_int, B_int):
            acc = pk.encrypt(int(b_int))
            for enc_feat, wij in zip(enc_x, wrow):
                # enc_feat * wij -> scalar multiplication supported (ciphertext * plaintext int)
                acc = acc + (enc_feat * int(wij))
            dec_int = sk.decrypt(acc)
            # decode back to float (divide by scale^2)
            score = float(dec_int) / (scale ** 2)
            cls_scores.append(score)
        logits.append(cls_scores)
    wall = time.time() - t0
    return np.array(logits), wall

# CKKS single-sample inference (encrypt each sample)
def ckks_single_inference_multiclass(X, W, B, ctx_scale_exp=CKKS_CTX_SCALE_EXP):
    """
    X: (n, d)
    W: (n_classes, d)
    B: (n_classes,)
    returns: logits (n, n_classes), wall_time
    """
    ctx = ts.context(ts.SCHEME_TYPE.CKKS, poly_modulus_degree=8192,
                     coeff_mod_bit_sizes=[60, 40, 40, 60])
    ctx.global_scale = 2 ** ctx_scale_exp
    ctx.generate_galois_keys()
    enc_w_per_class = [ts.ckks_vector(ctx, w.tolist()) for w in W]
    logits = []
    t0 = time.time()
    for x in X:
        enc_x = ts.ckks_vector(ctx, x.tolist())
        scores = []
        for enc_w, b in zip(enc_w_per_class, B.tolist()):
            # enc_x.dot(enc_w) returns a CKKS vector (length 1 or scalar-like)
            enc_dot = enc_x.dot(enc_w)
            dec_dot = enc_dot.decrypt()
            # decrypt returns list-like; take first element if list
            val = float(dec_dot[0]) if hasattr(dec_dot, "__len__") else float(dec_dot)
            scores.append(val + float(b))
        logits.append(scores)
    wall = time.time() - t0
    return np.array(logits), wall

# CKKS batched inference: pack each feature column across small batch as ckks_vector
def ckks_batched_inference_multiclass(X, W, B, batch_size=CKKS_BATCH_SIZE, ctx_scale_exp=CKKS_CTX_SCALE_EXP):
    """
    X: (n, d)
    W: (n_classes, d)
    B: (n_classes,)
    returns: logits (n, n_classes), wall_time
    """
    ctx = ts.context(ts.SCHEME_TYPE.CKKS, poly_modulus_degree=8192,
                     coeff_mod_bit_sizes=[60, 40, 40, 60])
    ctx.global_scale = 2 ** ctx_scale_exp
    ctx.generate_galois_keys()
    n = X.shape[0]
    logits_all = []
    t0 = time.time()
    for start in range(0, n, batch_size):
        end = min(start + batch_size, n)
        batch = X[start:end]  # (bs, d)
        bs = batch.shape[0]
        # enc_logits_per_class holds encrypted vectors (length bs) for each class
        enc_logits_per_class = [None] * W.shape[0]
        for j in range(batch.shape[1]):
            col_vals = batch[:, j].astype(float).tolist()  # length bs
            enc_col = ts.ckks_vector(ctx, col_vals)
            for cls in range(W.shape[0]):
                wj = float(W[cls, j])
                term = enc_col * wj  # multiply by plaintext scalar
                if enc_logits_per_class[cls] is None:
                    enc_logits_per_class[cls] = term
                else:
                    enc_logits_per_class[cls] = enc_logits_per_class[cls] + term
        # add bias vector per class
        for cls in range(W.shape[0]):
            bias_vec = ts.ckks_vector(ctx, [float(B[cls])] * bs)
            enc_logits_per_class[cls] = enc_logits_per_class[cls] + bias_vec
        # decrypt per-class vectors and append per-sample scores
        decs = [enc.decrypt() for enc in enc_logits_per_class]  # list length n_classes, each is list length bs
        for i in range(bs):
            sample_scores = [float(decs[cls][i]) for cls in range(W.shape[0])]
            logits_all.append(sample_scores)
    wall = time.time() - t0
    return np.array(logits_all), wall

# ---------- helper metrics ----------
def softmax_rows(logits):
    ex = np.exp(logits - np.max(logits, axis=1, keepdims=True))
    return ex / ex.sum(axis=1, keepdims=True)

def metrics_from_probs(probs, y_true):
    preds = np.argmax(probs, axis=1)
    return {
        "accuracy": accuracy_score(y_true, preds),
        "precision_macro": precision_score(y_true, preds, average="macro", zero_division=0),
        "recall_macro": recall_score(y_true, preds, average="macro", zero_division=0),
        "f1_macro": f1_score(y_true, preds, average="macro", zero_division=0),
        "roc_auc_ovr": roc_auc_score(y_true, probs, multi_class="ovr"),
        "log_loss": log_loss(y_true, probs)
    }

def logits_to_probs_multiclass(logits):
    logits = np.array(logits)
    if logits.ndim == 1:
        logits = logits.reshape(1, -1)
    return softmax_rows(logits)

# ---------- Main CV loop with real HE inference ----------
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RND)
fold_results = []
fold = 0

for train_idx, test_idx in skf.split(np.arange(len(embeddings)), labels):
    fold += 1
    print(f"\n=== Fold {fold}/{N_FOLDS} ===")
    X_train, y_train = embeddings[train_idx], labels[train_idx]
    X_test, y_test = embeddings[test_idx], labels[test_idx]
    print("train/test sizes:", X_train.shape[0], X_test.shape[0])

    # Train logistic regression (plaintext baseline) on train
    clf = LogisticRegression(max_iter=1000, solver="lbfgs", multi_class="multinomial")
    clf.fit(X_train, y_train)

    # Plaintext prediction on test
    probs_plain = clf.predict_proba(X_test)
    metrics_plain = metrics_from_probs(probs_plain, y_test)
    # Prepare weights & bias for HE
    W = clf.coef_.astype(float)    # shape (n_classes, d)
    B = clf.intercept_.astype(float)

    # Paillier (real encrypted inference, per-sample)
    paillier_logits, paillier_time = paillier_inference_multiclass(X_test, W, B, scale=PAILLIER_SCALE, key_size=PAILLIER_KEY)
    probs_paillier = logits_to_probs_multiclass(paillier_logits)
    metrics_paillier = metrics_from_probs(probs_paillier, y_test)

    # CKKS single (encrypt each sample, dot with plaintext weight vectors)
    ckks_single_logits, ckks_single_time = ckks_single_inference_multiclass(X_test, W, B, ctx_scale_exp=CKKS_CTX_SCALE_EXP)
    probs_ckks_single = logits_to_probs_multiclass(ckks_single_logits)
    metrics_ckks_single = metrics_from_probs(probs_ckks_single, y_test)

    # CKKS batched (pack small batches)
    ckks_batch_logits, ckks_batch_time = ckks_batched_inference_multiclass(X_test, W, B, batch_size=CKKS_BATCH_SIZE, ctx_scale_exp=CKKS_CTX_SCALE_EXP)
    probs_ckks_batch = logits_to_probs_multiclass(ckks_batch_logits)
    metrics_ckks_batch = metrics_from_probs(probs_ckks_batch, y_test)

    print("Fold metrics (Plain / Paillier / CKKS single / CKKS batched):")
    print("Plain:", metrics_plain)
    print("Paillier:", metrics_paillier)
    print("CKKS single:", metrics_ckks_single)
    print("CKKS batched:", metrics_ckks_batch)
    print("Runtimes (s): paillier=%.2f, ckks_single=%.2f, ckks_batch=%.2f" % (paillier_time, ckks_single_time, ckks_batch_time))

    fold_results.append({
        "plain": metrics_plain,
        "paillier": metrics_paillier,
        "ckks_single": metrics_ckks_single,
        "ckks_batch": metrics_ckks_batch,
        "time_paillier": paillier_time,
        "time_ckks_single": ckks_single_time,
        "time_ckks_batch": ckks_batch_time
    })

# ---------- aggregate results across folds ----------
def agg(key):
    arr = [fr[key] for fr in fold_results]
    df = pd.DataFrame(arr)
    mean = df.mean().to_dict(); std = df.std().to_dict()
    return {k: f"{mean[k]:.4f} ± {std[k]:.4f}" for k in mean.keys()}

print("\n=== Aggregated (mean ± std) ===")
print("Plaintext:", agg('plain'))
print("Paillier :", agg('paillier'))
print("CKKS sing:", agg('ckks_single'))
print("CKKS bat :", agg('ckks_batch'))

# runtimes
times = pd.DataFrame([{"paillier":fr["time_paillier"], "ckks_single":fr["time_ckks_single"], "ckks_batch":fr["time_ckks_batch"]} for fr in fold_results])
print("\nAverage runtimes (s):", times.mean().to_dict())

print("\nNotes:")
print("- Paillier is exact (integer encoding) but typically slow (per-sample encryption & decrypt).")
print("- CKKS single encrypts each sample and computes dot-products with plaintext weight vectors.")
print("- CKKS batched packs features across a small batch to amortize cost; it's usually fastest among CKKS options.")
print("- If runtime is too long, reduce N_FOLDS, GIN_EPOCHS, or GIN_EMB.")


Device: cpu


Processing...
Done!


GIN epoch 5/15, loss=2.8925
GIN epoch 10/15, loss=2.2196
GIN epoch 15/15, loss=1.7999
Embeddings shape: (2708, 64)

=== Fold 1/3 ===
train/test sizes: 1805 903


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Fold metrics (Plain / Paillier / CKKS single / CKKS batched):
Plain: {'accuracy': 0.858250276854928, 'precision_macro': 0.8802489031265378, 'recall_macro': 0.836923039188103, 'f1_macro': 0.8555365059531822, 'roc_auc_ovr': np.float64(0.9684422895651538), 'log_loss': 0.5887622220512405}
Paillier: {'accuracy': 0.858250276854928, 'precision_macro': 0.8802489031265378, 'recall_macro': 0.836923039188103, 'f1_macro': 0.8555365059531822, 'roc_auc_ovr': np.float64(0.9684423973546368), 'log_loss': 0.5887688696859924}
CKKS single: {'accuracy': 0.858250276854928, 'precision_macro': 0.8802489031265378, 'recall_macro': 0.836923039188103, 'f1_macro': 0.8555365059531822, 'roc_auc_ovr': np.float64(0.9684412132624154), 'log_loss': 0.588762224108625}
CKKS batched: {'accuracy': 0.858250276854928, 'precision_macro': 0.8802489031265378, 'recall_macro': 0.836923039188103, 'f1_macro': 0.8555365059531822, 'roc_auc_ovr': np.float64(0.968443365867892), 'log_loss': 0.588762225340809}
Runtimes (s): paillier=11733.

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Fold metrics (Plain / Paillier / CKKS single / CKKS batched):
Plain: {'accuracy': 0.8549280177187154, 'precision_macro': 0.8608861138733361, 'recall_macro': 0.8323137158347702, 'f1_macro': 0.8447612473499024, 'roc_auc_ovr': np.float64(0.9746501538466521), 'log_loss': 0.5229538908199126}
Paillier: {'accuracy': 0.8549280177187154, 'precision_macro': 0.8608861138733361, 'recall_macro': 0.8323137158347702, 'f1_macro': 0.8447612473499024, 'roc_auc_ovr': np.float64(0.9746488318542508), 'log_loss': 0.5229471132928968}
CKKS single: {'accuracy': 0.8549280177187154, 'precision_macro': 0.8608861138733361, 'recall_macro': 0.8323137158347702, 'f1_macro': 0.8447612473499024, 'roc_auc_ovr': np.float64(0.9746501538466521), 'log_loss': 0.5229538829760233}
CKKS batched: {'accuracy': 0.8549280177187154, 'precision_macro': 0.8608861138733361, 'recall_macro': 0.8323137158347702, 'f1_macro': 0.8447612473499024, 'roc_auc_ovr': np.float64(0.9746501538466521), 'log_loss': 0.5229538826649586}
Runtimes (s): pail

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Fold metrics (Plain / Paillier / CKKS single / CKKS batched):
Plain: {'accuracy': 0.8503325942350333, 'precision_macro': 0.858534893986112, 'recall_macro': 0.8245466008722014, 'f1_macro': 0.8399271689854656, 'roc_auc_ovr': np.float64(0.9740611549057722), 'log_loss': 0.5299500242596352}
Paillier: {'accuracy': 0.8503325942350333, 'precision_macro': 0.858534893986112, 'recall_macro': 0.8245466008722014, 'f1_macro': 0.8399271689854656, 'roc_auc_ovr': np.float64(0.9740616579823833), 'log_loss': 0.5299363752253867}
CKKS single: {'accuracy': 0.8503325942350333, 'precision_macro': 0.858534893986112, 'recall_macro': 0.8245466008722014, 'f1_macro': 0.8399271689854656, 'roc_auc_ovr': np.float64(0.9740611549057719), 'log_loss': 0.529950021333019}
CKKS batched: {'accuracy': 0.8503325942350333, 'precision_macro': 0.858534893986112, 'recall_macro': 0.8245466008722014, 'f1_macro': 0.8399271689854656, 'roc_auc_ovr': np.float64(0.9740611549057719), 'log_loss': 0.5299500204486999}
Runtimes (s): paillier=

# **GCN Model**

In [ ]:
# =============================================================================
# Cora node-level pipeline — GCN -> LogisticRegressionCV -> evaluate:
#   Plaintext, Paillier, CKKS (single & batched) for multiclass
# =============================================================================

import time, random, math
import numpy as np
import torch
import torch.nn.functional as F
from tqdm import tqdm
from copy import deepcopy
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.linear_model import LogisticRegressionCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, log_loss
from sklearn.preprocessing import StandardScaler
from torch_geometric.datasets import Planetoid
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv
from torch.nn import BatchNorm1d, LayerNorm
import torch.nn as nn

import phe as paillier
import tenseal as ts
import pandas as pd

# ---------------- USER KNOBS ----------------
RND = 42
N_FOLDS = 5

# GCN params
GCN_HIDDEN = 256
GCN_EMB = 256
GCN_EPOCHS = 120
GCN_PATIENCE = 16
GCN_DROPOUT = 0.5
GCN_LR = 1e-3
GCN_WEIGHT_DECAY = 1e-4
GCN_GRAD_CLIP = 5.0
GCN_FEATURE_DROPOUT = 0.05

# Paillier tuning
PAILLIER_SCALE = 1000
PAILLIER_KEY = 1024

# CKKS tuning
CKKS_POLY = 4096
CKKS_CTX_SCALE_EXP = 20
CKKS_BATCH_SIZE = 8
CKKS_COEFFS = [40, 20, 40]

LR_AUGMENT_COPIES = 2
LR_AUG_NOISE_STD_REL = 0.01

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
device = torch.device(DEVICE)
print("Device:", device)

# ---------------- reproducibility ----------------
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(RND)

# ---------- Node-level GCN model (produces node embeddings) ----------
class GCNNodeModel(torch.nn.Module):
    def __init__(self, in_ch, hidden=GCN_HIDDEN, emb_dim=GCN_EMB, dropout=GCN_DROPOUT, n_layers=3):
        super().__init__()
        self.n_layers = n_layers
        self.convs = nn.ModuleList()
        self.norms = nn.ModuleList()

        self.convs.append(GCNConv(in_ch, hidden))
        self.norms.append(BatchNorm1d(hidden))

        for _ in range(1, n_layers):
            self.convs.append(GCNConv(hidden, hidden))
            self.norms.append(LayerNorm(hidden))

        self.lin1 = nn.Linear(hidden, emb_dim)
        self.dropout = nn.Dropout(dropout)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            if isinstance(m, GCNConv):
                try:
                    if hasattr(m, 'lin') and hasattr(m.lin, 'weight'):
                        nn.init.xavier_uniform_(m.lin.weight)
                except Exception:
                    pass

    def forward(self, x, edge_index, feat_dropout=0.0):
        if self.training and feat_dropout > 0:
            mask = (torch.rand_like(x) > feat_dropout).float()
            x = x * mask

        residual = None
        for i in range(self.n_layers):
            h = self.convs[i](x, edge_index)
            h = self.norms[i](h)
            h = F.gelu(h)
            if residual is not None and residual.shape == h.shape:
                h = h + residual
            residual = h
            x = F.dropout(h, p=self.dropout.p if isinstance(self.dropout, nn.Dropout) else 0.5, training=self.training)

        emb = F.gelu(self.lin1(x))
        emb = self.dropout(emb)
        return emb  # shape: [num_nodes, emb_dim]

    def extract_emb(self, data):
        self.eval()
        with torch.no_grad():
            data = data.to(device)
            emb = self.forward(data.x.float(), data.edge_index, feat_dropout=0.0)
            return emb.cpu().numpy()

# ---------- helpers: training GCN for node classification ----------
def train_gcn_node(data, train_idx, val_idx, num_classes, epochs=GCN_EPOCHS, patience=GCN_PATIENCE, lr=GCN_LR, weight_decay=GCN_WEIGHT_DECAY):
    model = GCNNodeModel(data.num_features).to(device)
    head = torch.nn.Linear(GCN_EMB, num_classes).to(device)

    opt = torch.optim.AdamW(list(model.parameters()) + list(head.parameters()), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='min', factor=0.5, patience=6, min_lr=1e-6)
    crit = torch.nn.CrossEntropyLoss()

    best_val = float("inf"); wait = 0; best_state = None
    # We'll create simple loaders: in this node-level setup we'll process whole graph but compute loss only on selected nodes
    for ep in range(1, epochs+1):
        model.train(); head.train()
        opt.zero_grad()
        data = data.to(device)
        emb = model(data.x.float(), data.edge_index, feat_dropout=GCN_FEATURE_DROPOUT)  # [N, emb]
        out = head(emb)  # [N, C]
        loss = crit(out[train_idx], data.y[train_idx])
        loss.backward()
        torch.nn.utils.clip_grad_norm_(list(model.parameters()) + list(head.parameters()), max_norm=GCN_GRAD_CLIP)
        opt.step()

        # Validation
        model.eval(); head.eval()
        with torch.no_grad():
            emb = model(data.x.float(), data.edge_index, feat_dropout=0.0)
            out = head(emb)
            vloss = crit(out[val_idx], data.y[val_idx]).item()

        scheduler.step(vloss)

        if vloss < best_val - 1e-6:
            best_val = vloss; wait = 0
            best_state = (deepcopy(model.state_dict()), deepcopy(head.state_dict()))
        else:
            wait += 1

        if ep % 5 == 0 or ep == 1:
            print(f"Epoch {ep:03d} — train_loss: {loss.item():.4f}, val_loss: {vloss:.4f}, lr: {opt.param_groups[0]['lr']:.6f}, wait: {wait}")

        if wait >= patience:
            print(f"Early stopping at epoch {ep} (wait={wait} >= patience={patience})")
            break

    if best_state is not None:
        model.load_state_dict(best_state[0])
        head.load_state_dict(best_state[1])
    return model

# ---------- metrics (multiclass) ----------
def softmax_rows(logits):
    # logits: [n_samples, n_classes]
    l = np.array(logits, dtype=float)
    l = l - np.max(l, axis=1, keepdims=True)
    ex = np.exp(l)
    return ex / np.sum(ex, axis=1, keepdims=True)

def metrics_from_probs_multiclass(probs, y_true):
    preds = np.argmax(probs, axis=1)
    try:
        roc = roc_auc_score(y_true, probs, multi_class='ovr')
    except Exception:
        roc = float("nan")
    return {
        "accuracy": accuracy_score(y_true, preds),
        "precision_macro": precision_score(y_true, preds, average='macro', zero_division=0),
        "recall_macro": recall_score(y_true, preds, average='macro', zero_division=0),
        "f1_macro": f1_score(y_true, preds, average='macro', zero_division=0),
        "roc_auc_ovr": roc,
        "log_loss": log_loss(y_true, probs)
    }

# ---------- Paillier multiclass inference ----------
def paillier_inference_multiclass(X, W, b, scale=PAILLIER_SCALE, key_size=PAILLIER_KEY):
    # X: [n_samples, n_features]
    # W: [n_classes, n_features]
    # b: [n_classes]
    pk, sk = paillier.generate_paillier_keypair(n_length=key_size)
    # integerize weights
    W_int = [[int(round(float(wij) * scale)) for wij in w_row] for w_row in W.tolist()]
    b_int = [int(round(float(bc) * (scale ** 2))) for bc in b.tolist()]

    logits = []
    t0 = time.time()
    for x in X:
        x_int = [int(round(float(xj) * scale)) for xj in x.tolist()]
        enc_x = [pk.encrypt(int(xj)) for xj in x_int]
        scores = []
        for class_idx, w_int_row in enumerate(W_int):
            acc = pk.encrypt(int(b_int[class_idx]))
            for enc_feat, wi in zip(enc_x, w_int_row):
                acc = acc + (enc_feat * int(wi))
            dec_int = sk.decrypt(acc)
            score = float(dec_int) / (scale ** 2)
            scores.append(score)
        logits.append(scores)
    wall = time.time() - t0
    return np.array(logits), wall  # shape [n_samples, n_classes]

# ---------- CKKS single multiclass inference ----------
def ckks_single_inference_multiclass(X, W, b, poly_mod=CKKS_POLY, ctx_scale_exp=CKKS_CTX_SCALE_EXP, coeff_mod_bit_sizes=CKKS_COEFFS):
    # W: [n_classes, n_features], b: [n_classes]
    ctx = ts.context(ts.SCHEME_TYPE.CKKS, poly_modulus_degree=poly_mod, coeff_mod_bit_sizes=coeff_mod_bit_sizes)
    ctx.global_scale = 2 ** ctx_scale_exp
    ctx.generate_galois_keys()
    enc_w_list = [ts.ckks_vector(ctx, w_row.tolist()) for w_row in W]
    logits = []
    t0 = time.time()
    for x in X:
        enc_x = ts.ckks_vector(ctx, x.tolist())
        scores = []
        for enc_w in enc_w_list:
            enc_dot = enc_x.dot(enc_w)
            dec_dot = enc_dot.decrypt()
            val = float(dec_dot[0]) if hasattr(dec_dot, "__len__") else float(dec_dot)
            scores.append(val)
        scores = np.array(scores) + b
        logits.append(scores.tolist())
    wall = time.time() - t0
    return np.array(logits), wall

# ---------- CKKS batched multiclass inference ----------
def ckks_batched_inference_multiclass(X, W, b, batch_size=CKKS_BATCH_SIZE, poly_mod=CKKS_POLY, ctx_scale_exp=CKKS_CTX_SCALE_EXP, coeff_mod_bit_sizes=CKKS_COEFFS):
    ctx = ts.context(ts.SCHEME_TYPE.CKKS, poly_modulus_degree=poly_mod, coeff_mod_bit_sizes=coeff_mod_bit_sizes)
    ctx.global_scale = 2 ** ctx_scale_exp
    ctx.generate_galois_keys()
    enc_w_list = [ts.ckks_vector(ctx, w_row.tolist()) for w_row in W]
    n = X.shape[0]
    logits = []
    t0 = time.time()
    for start in range(0, n, batch_size):
        end = min(start + batch_size, n)
        batch = X[start:end]
        enc_batch = [ts.ckks_vector(ctx, x.tolist()) for x in batch]
        for enc_x in enc_batch:
            scores = []
            for enc_w in enc_w_list:
                enc_dot = enc_x.dot(enc_w)
                dec_dot = enc_dot.decrypt()
                val = float(dec_dot[0]) if hasattr(dec_dot, "__len__") else float(dec_dot)
                scores.append(val)
            scores = np.array(scores) + b
            logits.append(scores.tolist())
    wall = time.time() - t0
    return np.array(logits), wall

# ---------------- main CV loop (on nodes) ----------------
print("Loading Cora dataset...")
dataset = Planetoid(root="/tmp/Cora_planetoid", name="Cora")
data = dataset[0]
n_nodes = data.num_nodes
labels = data.y.cpu().numpy()
num_classes = dataset.num_classes
print(f"Cora loaded — nodes: {n_nodes}, features: {data.num_features}, classes: {num_classes}")

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RND)

fold_results = []
fold = 0
for trainval_idx, test_idx in skf.split(np.arange(n_nodes), labels):
    fold += 1
    print(f"\n=== Fold {fold}/{N_FOLDS} ===")
    # split trainval into train/val
    train_idx, val_idx = train_test_split(trainval_idx, test_size=0.125, random_state=RND, stratify=labels[trainval_idx])
    print("train/val/test sizes (nodes):", len(train_idx), len(val_idx), len(test_idx))

    gcn = train_gcn_node(data, train_idx=torch.tensor(train_idx, dtype=torch.long).to(device),
                         val_idx=torch.tensor(val_idx, dtype=torch.long).to(device),
                         num_classes=num_classes,
                         epochs=GCN_EPOCHS, patience=GCN_PATIENCE)

    # extract embeddings for all nodes
    X_all = gcn.extract_emb(data)  # [n_nodes, emb_dim]
    y_all = data.y.cpu().numpy()

    # split embeddings according to indices
    X_train = X_all[train_idx]; y_train = y_all[train_idx]
    X_val   = X_all[val_idx];   y_val   = y_all[val_idx]
    X_test  = X_all[test_idx];  y_test  = y_all[test_idx]
    print("Emb shapes (train/val/test):", X_train.shape, X_val.shape, X_test.shape)

    scaler = StandardScaler()
    scaler.fit(np.vstack([X_train, X_val]))
    X_train_s = scaler.transform(X_train)
    X_val_s   = scaler.transform(X_val)
    X_test_s  = scaler.transform(X_test)

    # augmentation (same as before)
    def augment(X, y, copies=LR_AUGMENT_COPIES, noise_std_rel=LR_AUG_NOISE_STD_REL):
        Xs, ys = [X], [y]
        feat_std = np.std(X, axis=0, ddof=0)
        feat_std = np.where(feat_std == 0.0, 1.0, feat_std)
        for _ in range(copies):
            noise = np.random.normal(scale=noise_std_rel * feat_std, size=X.shape)
            Xs.append(X + noise)
            ys.append(y.copy())
        return np.vstack(Xs), np.concatenate(ys)

    X_aug, y_aug = augment(np.vstack([X_train_s, X_val_s]), np.concatenate([y_train, y_val]))

    # LogisticRegressionCV (multiclass)
    clf = LogisticRegressionCV(
        Cs=5,
        cv=3,
        penalty='l2',
        scoring='f1_macro',
        solver='lbfgs',
        multi_class='multinomial',
        max_iter=10000,
        class_weight='balanced',
        n_jobs=1,
        refit=True
    )
    clf.fit(X_aug, y_aug)

    probs_plain = clf.predict_proba(X_test_s)
    metrics_plain = metrics_from_probs_multiclass(probs_plain, y_test)

    logits_plain = clf.decision_function(X_test_s)  # [n_samples, n_classes]
    W = clf.coef_.astype(float)        # shape [n_classes, n_features]
    b = clf.intercept_.astype(float)   # shape [n_classes]

    # Paillier multiclass
    paillier_logits, paillier_time = paillier_inference_multiclass(X_test_s, W, b)
    probs_paillier = softmax_rows(paillier_logits)
    metrics_paillier = metrics_from_probs_multiclass(probs_paillier, y_test)

    # CKKS single multiclass
    ckks_single_logits, ckks_single_time = ckks_single_inference_multiclass(X_test_s, W, b)
    probs_ckks_single = softmax_rows(ckks_single_logits)
    metrics_ckks_single = metrics_from_probs_multiclass(probs_ckks_single, y_test)

    # CKKS batched multiclass
    ckks_batch_logits, ckks_batch_time = ckks_batched_inference_multiclass(X_test_s, W, b)
    probs_ckks_batch = softmax_rows(ckks_batch_logits)
    metrics_ckks_batch = metrics_from_probs_multiclass(probs_ckks_batch, y_test)

    print("Fold results (Plain / Paillier / CKKS single / CKKS batched):")
    print("Plain:", metrics_plain)
    print("Paillier:", metrics_paillier)
    print("CKKS single:", metrics_ckks_single)
    print("CKKS batched:", metrics_ckks_batch)
    print("Runtimes (s): paillier=%.2f, ckks_single=%.2f, ckks_batch=%.2f" % (paillier_time, ckks_single_time, ckks_batch_time))

    fold_results.append({
        "plain": metrics_plain, "paillier": metrics_paillier,
        "ckks_single": metrics_ckks_single, "ckks_batch": metrics_ckks_batch,
        "time_paillier": paillier_time, "time_ckks_single": ckks_single_time, "time_ckks_batch": ckks_batch_time
    })

# ---------- aggregate results ----------
import pandas as pd

def agg(key):
    arr = [fr[key] for fr in fold_results]
    df = pd.DataFrame(arr)
    mean = df.mean().to_dict(); std = df.std().to_dict()
    return {k: f"{mean[k]:.4f} ± {std[k]:.4f}" for k in mean.keys()}

print("\n=== Aggregated (mean ± std) ===")
print("Plaintext:", agg("plain"))
print("Paillier :", agg("paillier"))
print("CKKS sing:", agg("ckks_single"))
print("CKKS bat :", agg("ckks_batch"))

times = pd.DataFrame([{"paillier":fr["time_paillier"], "ckks_single":fr["time_ckks_single"], "ckks_batch":fr["time_ckks_batch"]} for fr in fold_results])
print("\nAverage runtimes (s):", times.mean().to_dict())



Device: cpu
Loading Cora dataset...
Cora loaded — nodes: 2708, features: 1433, classes: 7

=== Fold 1/5 ===
train/val/test sizes (nodes): 1895 271 542
Epoch 001 — train_loss: 2.2051, val_loss: 1.7187, lr: 0.001000, wait: 0
Epoch 005 — train_loss: 1.3953, val_loss: 1.1567, lr: 0.001000, wait: 0
Epoch 010 — train_loss: 0.7718, val_loss: 0.5480, lr: 0.001000, wait: 0
Epoch 015 — train_loss: 0.4636, val_loss: 0.3132, lr: 0.001000, wait: 0
Epoch 020 — train_loss: 0.3177, val_loss: 0.2476, lr: 0.001000, wait: 0
Epoch 025 — train_loss: 0.2430, val_loss: 0.2474, lr: 0.001000, wait: 2
Epoch 030 — train_loss: 0.2088, val_loss: 0.2645, lr: 0.000500, wait: 7
Epoch 035 — train_loss: 0.1837, val_loss: 0.2753, lr: 0.000500, wait: 12
Early stopping at epoch 39 (wait=16 >= patience=16)
Emb shapes (train/val/test): (1895, 256) (271, 256) (542, 256)


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1908: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Fold results (Plain / Paillier / CKKS single / CKKS batched):
Plain: {'accuracy': 0.8505535055350554, 'precision_macro': 0.8347104030727704, 'recall_macro': 0.8375649560258969, 'f1_macro': 0.8356513313922868, 'roc_auc_ovr': np.float64(0.9696861458200191), 'log_loss': 2.1494680836960285}
Paillier: {'accuracy': 0.8505535055350554, 'precision_macro': 0.8347104030727704, 'recall_macro': 0.8375649560258969, 'f1_macro': 0.8356513313922868, 'roc_auc_ovr': np.float64(0.9696708646008687), 'log_loss': 2.1496576541008756}
CKKS single: {'accuracy': 0.8505535055350554, 'precision_macro': 0.8347104030727704, 'recall_macro': 0.8375649560258969, 'f1_macro': 0.8356513313922868, 'roc_auc_ovr': np.float64(0.9696543302599913), 'log_loss': 2.177048410421301}
CKKS batched: {'accuracy': 0.8505535055350554, 'precision_macro': 0.8347104030727704, 'recall_macro': 0.8375649560258969, 'f1_macro': 0.8356513313922868, 'roc_auc_ovr': np.float64(0.9696939877231301), 'log_loss': 2.1771316709824258}
Runtimes (s): paill

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1908: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Fold results (Plain / Paillier / CKKS single / CKKS batched):
Plain: {'accuracy': 0.8690036900369004, 'precision_macro': 0.8695520380354255, 'recall_macro': 0.8431962688648914, 'f1_macro': 0.8538414819091146, 'roc_auc_ovr': np.float64(0.9711339176311896), 'log_loss': 1.8885855044205773}
Paillier: {'accuracy': 0.8690036900369004, 'precision_macro': 0.8695520380354255, 'recall_macro': 0.8431962688648914, 'f1_macro': 0.8538414819091146, 'roc_auc_ovr': np.float64(0.9711479307048189), 'log_loss': 1.8887484767172327}
CKKS single: {'accuracy': 0.8671586715867159, 'precision_macro': 0.8679696086564883, 'recall_macro': 0.8423251887255186, 'f1_macro': 0.8526271744694107, 'roc_auc_ovr': np.float64(0.9710851963630959), 'log_loss': 1.9062382388457204}
CKKS batched: {'accuracy': 0.8690036900369004, 'precision_macro': 0.8695520380354255, 'recall_macro': 0.8431962688648914, 'f1_macro': 0.8538414819091146, 'roc_auc_ovr': np.float64(0.9711691110631419), 'log_loss': 1.9048009761050577}
Runtimes (s): pail

...

Fold results (Plain / Paillier / CKKS single / CKKS batched):
Plain: {'accuracy': 0.8579335793357934, 'precision_macro': 0.8419737687794988, 'recall_macro': 0.8415943537558718, 'f1_macro': 0.8409895909723969, 'roc_auc_ovr': np.float64(0.9771816794421283), 'log_loss': 1.8024668193257376}
Paillier: {'accuracy': 0.8579335793357934, 'precision_macro': 0.8419737687794988, 'recall_macro': 0.8415943537558718, 'f1_macro': 0.8409895909723969, 'roc_auc_ovr': np.float64(0.9771742985104321), 'log_loss': 1.8028033504861054}
CKKS single: {'accuracy': 0.8579335793357934, 'precision_macro': 0.8419737687794988, 'recall_macro': 0.8415943537558718, 'f1_macro': 0.8409895909723969, 'roc_auc_ovr': np.float64(0.9771741217132837), 'log_loss': 1.8242466810436004}
CKKS batched: {'accuracy': 0.8579335793357934, 'precision_macro': 0.8419737687794988, 'recall_macro': 0.8415943537558718, 'f1_macro': 0.8409895909723969, 'roc_auc_ovr': np.float64(0.9772240574164973), 'log_loss': 1.8236733308463404}
Runtimes (s): pail

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1908: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Fold results (Plain / Paillier / CKKS single / CKKS batched):
Plain: {'accuracy': 0.8410351201478743, 'precision_macro': 0.835262051573417, 'recall_macro': 0.8294143366657114, 'f1_macro': 0.8315049110791372, 'roc_auc_ovr': np.float64(0.9639200862541232), 'log_loss': 2.324728212715604}
Paillier: {'accuracy': 0.8410351201478743, 'precision_macro': 0.835262051573417, 'recall_macro': 0.8294143366657114, 'f1_macro': 0.8315049110791372, 'roc_auc_ovr': np.float64(0.9639336978884102), 'log_loss': 2.3246599840613507}
CKKS single: {'accuracy': 0.8410351201478743, 'precision_macro': 0.835262051573417, 'recall_macro': 0.8294143366657114, 'f1_macro': 0.8315049110791372, 'roc_auc_ovr': np.float64(0.9639162375258764), 'log_loss': 2.353929048567207}
CKKS batched: {'accuracy': 0.8410351201478743, 'precision_macro': 0.835262051573417, 'recall_macro': 0.8294143366657114, 'f1_macro': 0.8315049110791372, 'roc_auc_ovr': np.float64(0.9638781572469195), 'log_loss': 2.3532704721325732}
Runtimes (s): paillier=3

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1908: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Fold results (Plain / Paillier / CKKS single / CKKS batched):
Plain: {'accuracy': 0.8632162661737524, 'precision_macro': 0.8503405806320536, 'recall_macro': 0.8530864368639453, 'f1_macro': 0.8511249145690767, 'roc_auc_ovr': np.float64(0.9746940574502542), 'log_loss': 1.827201533945965}
Paillier: {'accuracy': 0.8632162661737524, 'precision_macro': 0.8503405806320536, 'recall_macro': 0.8530864368639453, 'f1_macro': 0.8511249145690767, 'roc_auc_ovr': np.float64(0.974698557918595), 'log_loss': 1.8272801005542028}
CKKS single: {'accuracy': 0.8650646950092421, 'precision_macro': 0.8515946970140539, 'recall_macro': 0.8550985092985529, 'f1_macro': 0.8527514363234824, 'roc_auc_ovr': np.float64(0.9746739537170875), 'log_loss': 1.8473859806939552}
CKKS batched: {'accuracy': 0.8632162661737524, 'precision_macro': 0.8503405806320536, 'recall_macro': 0.8530864368639453, 'f1_macro': 0.8511249145690767, 'roc_auc_ovr': np.float64(0.9746978216737625), 'log_loss': 1.8470105322764139}
Runtimes (s): pailli

# **GAT Model**

In [ ]:
# =============================================================================
# Full pipeline — GAT embeddings -> LogisticRegression (OvR) -> evaluate:
#   Plaintext, Paillier (1024-bit + quantized), CKKS (poly=4096, scale=2**20, batched)
# Dataset: Cora (node classification). Paillier/CKKS use OvR for multiclass inference.
# =============================================================================

# If in Colab, uncomment installs (only if needed)
# !pip install -q scikit-learn phe tenseal tqdm
# PyG install is environment dependent; assume torch & torch_geometric are installed.

import time, random, math
import numpy as np
import torch
import torch.nn.functional as F
from copy import deepcopy
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, log_loss
from sklearn.preprocessing import StandardScaler
from torch_geometric.datasets import Planetoid
from torch_geometric.nn import GATConv
from torch.nn import BatchNorm1d
import torch.nn as nn

import phe as paillier
import tenseal as ts
import pandas as pd

# ---------------- USER KNOBS ----------------
RND = 42
N_FOLDS = 5

# GAT params
GAT_HIDDEN = 64
GAT_EMB = 64
GAT_EPOCHS = 200
GAT_PATIENCE = 20
GAT_HEADS = 4
GAT_DROPOUT = 0.6

# Paillier tuning (quantization and key size)
PAILLIER_SCALE = 1000        # quantization scale for floats -> integers
PAILLIER_KEY = 1024          # 1024-bit key for speed (use 2048+ for more security)

# CKKS tuning
CKKS_POLY = 4096
CKKS_CTX_SCALE_EXP = 20      # global_scale = 2**20
CKKS_BATCH_SIZE = 8
CKKS_COEFFS = [40, 20, 40]

LR_AUGMENT_COPIES = 2
LR_AUG_NOISE_STD = 1e-2

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# ---------------------------------------------

# reproducibility
random.seed(RND)
np.random.seed(RND)
torch.manual_seed(RND)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RND)
device = torch.device(DEVICE)
print("Device:", device)

# ---------- GAT model ----------
class GATModel(torch.nn.Module):
    def __init__(self, in_ch, hidden=GAT_HIDDEN, emb_dim=GAT_EMB, heads=GAT_HEADS, dropout=GAT_DROPOUT):
        super().__init__()
        # first layer outputs hidden * heads
        self.conv1 = GATConv(in_ch, hidden, heads=heads, dropout=dropout)
        self.bn1 = BatchNorm1d(hidden * heads)
        # second layer projects to embedding dim (using 1 head)
        self.conv2 = GATConv(hidden * heads, emb_dim, heads=1, concat=True, dropout=dropout)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, edge_index):
        x = F.elu(self.bn1(self.conv1(x, edge_index)))
        x = self.conv2(x, edge_index)
        x = self.dropout(x)
        return x  # node embeddings [num_nodes, emb_dim]

# ---------- helpers: training GAT ----------
def train_gat(data, train_idx, val_idx, epochs=GAT_EPOCHS, patience=GAT_PATIENCE):
    model = GATModel(data.num_features).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=5e-4)
    crit = torch.nn.CrossEntropyLoss()

    best_val = float("inf"); wait = 0; best_state = None
    for ep in range(1, epochs+1):
        model.train()
        opt.zero_grad()
        out = model(data.x.to(device), data.edge_index.to(device))  # [N, C_emb]
        loss = crit(out[train_idx], data.y[train_idx].to(device))
        loss.backward(); opt.step()

        # validation (use latest embeddings)
        model.eval()
        with torch.no_grad():
            out_eval = model(data.x.to(device), data.edge_index.to(device))
            vloss = crit(out_eval[val_idx], data.y[val_idx].to(device)).item()

        if vloss < best_val - 1e-4:
            best_val = vloss; wait = 0
            best_state = deepcopy(model.state_dict())
        else:
            wait += 1

        if ep % 10 == 0 or ep == 1:
            print(f"Epoch {ep:03d} train_loss: {loss.item():.4f} val_loss: {vloss:.4f} wait: {wait}")

        if wait >= patience:
            print(f"Early stopping at epoch {ep} (wait={wait} >= patience={patience})")
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model

# ---------- metrics (multiclass) ----------
def softmax_rows(logits):
    l = np.array(logits, dtype=float)
    l = l - np.max(l, axis=1, keepdims=True)
    ex = np.exp(l)
    return ex / np.sum(ex, axis=1, keepdims=True)

def metrics_from_probs_multiclass(probs, y_true):
    preds = np.argmax(probs, axis=1)
    try:
        roc = roc_auc_score(y_true, probs, multi_class='ovr')
    except Exception:
        roc = float("nan")
    return {
        "accuracy": float(accuracy_score(y_true, preds)),
        "precision_macro": float(precision_score(y_true, preds, average='macro', zero_division=0)),
        "recall_macro": float(recall_score(y_true, preds, average='macro', zero_division=0)),
        "f1_macro": float(f1_score(y_true, preds, average='macro', zero_division=0)),
        "roc_auc_ovr": float(roc) if not np.isnan(roc) else float("nan"),
        "log_loss": float(log_loss(y_true, probs))
    }

# ---------- Paillier multiclass (OvR) ----------
def paillier_inference_multiclass(X, W, b, scale=PAILLIER_SCALE, key_size=PAILLIER_KEY):
    """
    X: [n_samples, n_features]
    W: [n_classes, n_features]
    b: [n_classes]
    Returns: logits [n_samples, n_classes], wall_time
    """
    pk, sk = paillier.generate_paillier_keypair(n_length=key_size)
    # integerize weights
    W_int = [[int(round(float(wij) * scale)) for wij in w_row] for w_row in W.tolist()]
    b_int = [int(round(float(bc) * (scale ** 2))) for bc in b.tolist()]

    logits = []
    t0 = time.time()
    for x in X:
        # quantize x
        x_int = [int(round(float(xj) * scale)) for xj in x.tolist()]
        # encrypt features once per sample
        enc_x = [pk.encrypt(int(xj)) for xj in x_int]
        class_scores = []
        # compute encrypted dot for each class
        for class_idx, w_int_row in enumerate(W_int):
            acc = pk.encrypt(int(b_int[class_idx]))
            for enc_feat, wi in zip(enc_x, w_int_row):
                acc = acc + (enc_feat * int(wi))
            dec_int = sk.decrypt(acc)
            score = float(dec_int) / (scale ** 2)
            class_scores.append(score)
        logits.append(class_scores)
    wall = time.time() - t0
    return np.array(logits), wall

# ---------- CKKS single-sample multiclass ----------
def ckks_single_inference_multiclass(X, W, b, poly_mod=CKKS_POLY, ctx_scale_exp=CKKS_CTX_SCALE_EXP, coeff_mod_bit_sizes=CKKS_COEFFS):
    """
    For each sample we encrypt the sample as a CKKS vector and dot with each class weight vector.
    Returns logits [n_samples, n_classes]
    """
    ctx = ts.context(ts.SCHEME_TYPE.CKKS, poly_modulus_degree=poly_mod, coeff_mod_bit_sizes=coeff_mod_bit_sizes)
    ctx.global_scale = 2 ** ctx_scale_exp
    ctx.generate_galois_keys()

    # prepare encrypted/plain weight vectors (we'll use ckks_vector on weights so they are in same space)
    enc_w_list = [ts.ckks_vector(ctx, w_row.tolist()) for w_row in W]
    logits = []
    t0 = time.time()
    for x in X:
        enc_x = ts.ckks_vector(ctx, x.tolist())
        class_scores = []
        for enc_w in enc_w_list:
            enc_dot = enc_x.dot(enc_w)
            dec_dot = enc_dot.decrypt()
            # decrypt returns scalar-like or length-1 list
            val = float(dec_dot[0]) if hasattr(dec_dot, "__len__") else float(dec_dot)
            class_scores.append(val)
        # add bias vector (plaintext)
        class_scores = np.array(class_scores) + b
        logits.append(class_scores.tolist())
    wall = time.time() - t0
    return np.array(logits), wall

# ---------- CKKS batched multiclass inference ----------
def ckks_batched_inference_multiclass(X, W, b, batch_size=CKKS_BATCH_SIZE, poly_mod=CKKS_POLY, ctx_scale_exp=CKKS_CTX_SCALE_EXP, coeff_mod_bit_sizes=CKKS_COEFFS):
    ctx = ts.context(ts.SCHEME_TYPE.CKKS, poly_modulus_degree=poly_mod, coeff_mod_bit_sizes=coeff_mod_bit_sizes)
    ctx.global_scale = 2 ** ctx_scale_exp
    ctx.generate_galois_keys()
    enc_w_list = [ts.ckks_vector(ctx, w_row.tolist()) for w_row in W]
    n = X.shape[0]
    logits = []
    t0 = time.time()
    for start in range(0, n, batch_size):
        end = min(start + batch_size, n)
        batch = X[start:end]
        for x in batch:
            enc_x = ts.ckks_vector(ctx, x.tolist())
            class_scores = []
            for enc_w in enc_w_list:
                enc_dot = enc_x.dot(enc_w)
                dec_dot = enc_dot.decrypt()
                val = float(dec_dot[0]) if hasattr(dec_dot, "__len__") else float(dec_dot)
                class_scores.append(val)
            class_scores = np.array(class_scores) + b
            logits.append(class_scores.tolist())
    wall = time.time() - t0
    return np.array(logits), wall

# ---------- main CV loop ----------
print("Loading Cora dataset...")
dataset = Planetoid(root="/tmp/Cora_planetoid", name="Cora")
data = dataset[0]
n_nodes = data.num_nodes
labels = data.y.cpu().numpy()
num_classes = dataset.num_classes
print(f"Cora loaded — nodes: {n_nodes}, features: {data.num_features}, classes: {num_classes}")

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RND)

fold_results = []
fold = 0
for trainval_idx, test_idx in skf.split(np.arange(n_nodes), labels):
    fold += 1
    print(f"\n=== Fold {fold}/{N_FOLDS} ===")
    # split trainval into train/val
    train_idx, val_idx = train_test_split(trainval_idx, test_size=0.125, random_state=RND, stratify=labels[trainval_idx])
    # convert to tensors for GAT training indexing
    train_idx_t = torch.tensor(train_idx, dtype=torch.long)
    val_idx_t = torch.tensor(val_idx, dtype=torch.long)
    test_idx_t = torch.tensor(test_idx, dtype=torch.long)
    print("train/val/test sizes (nodes):", len(train_idx), len(val_idx), len(test_idx))

    # 1) train GAT on this split
    gat = train_gat(data, train_idx_t, val_idx_t, epochs=GAT_EPOCHS, patience=GAT_PATIENCE)

    # 2) extract node embeddings for all nodes
    gat.eval()
    with torch.no_grad():
        emb_all = gat(data.x.to(device), data.edge_index.to(device)).cpu().numpy()  # [n_nodes, emb_dim]

    X_train, y_train = emb_all[train_idx], labels[train_idx]
    X_val,   y_val   = emb_all[val_idx], labels[val_idx]
    X_test,  y_test  = emb_all[test_idx], labels[test_idx]
    print("Emb shapes (train/val/test):", X_train.shape, X_val.shape, X_test.shape)

    # 3) scale embeddings and augment
    scaler = StandardScaler()
    scaler.fit(np.vstack([X_train, X_val]))
    X_train_s = scaler.transform(X_train)
    X_val_s = scaler.transform(X_val)
    X_test_s = scaler.transform(X_test)

    def augment(X, y, copies=LR_AUGMENT_COPIES, noise_std=LR_AUG_NOISE_STD):
        Xs, ys = [X], [y]
        for _ in range(copies):
            Xs.append(X + np.random.normal(scale=noise_std, size=X.shape))
            ys.append(y.copy())
        return np.vstack(Xs), np.concatenate(ys)

    X_aug, y_aug = augment(np.vstack([X_train_s, X_val_s]), np.concatenate([y_train, y_val]))

    # 4) train logistic regression (OvR)
    clf = LogisticRegression(max_iter=10000, solver="lbfgs", class_weight="balanced", multi_class="ovr")
    clf.fit(X_aug, y_aug)

    # Plaintext evaluation (multiclass probabilities)
    probs_plain = clf.predict_proba(X_test_s)  # shape [n_samples, n_classes]
    metrics_plain = metrics_from_probs_multiclass(probs_plain, y_test)

    # Prepare weight matrix and bias vector (shape [n_classes, n_features], [n_classes])
    W = clf.coef_.astype(float)         # if binary, shape may be (1, d) with OvR; we handle that
    b = clf.intercept_.astype(float)

    # Ensure W has shape [n_classes, d]
    if W.ndim == 1:
        W = W.reshape(1, -1)

    # 5) Paillier multiclass OvR inference
    paillier_logits, paillier_time = paillier_inference_multiclass(X_test_s, W, b, scale=PAILLIER_SCALE, key_size=PAILLIER_KEY)
    probs_paillier = softmax_rows(paillier_logits)
    metrics_paillier = metrics_from_probs_multiclass(probs_paillier, y_test)

    # 6) CKKS single-sample multiclass
    ckks_single_logits, ckks_single_time = ckks_single_inference_multiclass(X_test_s, W, b, poly_mod=CKKS_POLY, ctx_scale_exp=CKKS_CTX_SCALE_EXP, coeff_mod_bit_sizes=CKKS_COEFFS)
    probs_ckks_single = softmax_rows(ckks_single_logits)
    metrics_ckks_single = metrics_from_probs_multiclass(probs_ckks_single, y_test)

    # 7) CKKS batched multiclass
    ckks_batch_logits, ckks_batch_time = ckks_batched_inference_multiclass(X_test_s, W, b, batch_size=CKKS_BATCH_SIZE, poly_mod=CKKS_POLY, ctx_scale_exp=CKKS_CTX_SCALE_EXP, coeff_mod_bit_sizes=CKKS_COEFFS)
    probs_ckks_batch = softmax_rows(ckks_batch_logits)
    metrics_ckks_batch = metrics_from_probs_multiclass(probs_ckks_batch, y_test)

    print("Fold results (Plain / Paillier / CKKS single / CKKS batched):")
    print("Plain:", metrics_plain)
    print("Paillier:", metrics_paillier)
    print("CKKS single:", metrics_ckks_single)
    print("CKKS batched:", metrics_ckks_batch)
    print("Runtimes (s): paillier=%.2f, ckks_single=%.2f, ckks_batch=%.2f" % (paillier_time, ckks_single_time, ckks_batch_time))

    fold_results.append({
        "plain": metrics_plain, "paillier": metrics_paillier,
        "ckks_single": metrics_ckks_single, "ckks_batch": metrics_ckks_batch,
        "time_paillier": paillier_time, "time_ckks_single": ckks_single_time, "time_ckks_batch": ckks_batch_time
    })

# ---------- aggregate results ----------
# aggregate
import pandas as pd

def agg(key):
    arr = [fr[key] for fr in fold_results]
    df = pd.DataFrame(arr)
    mean = df.mean().to_dict(); std = df.std().to_dict()
    return {k: f"{mean[k]:.4f} ± {std[k]:.4f}" for k in mean.keys()}

print("\n=== Aggregated (mean ± std) ===")
print("Plaintext:", agg("plain"))
print("Paillier :", agg("paillier"))
print("CKKS sing:", agg("ckks_single"))
print("CKKS bat :", agg("ckks_batch"))

times = pd.DataFrame([{"paillier":fr["time_paillier"], "ckks_single":fr["time_ckks_single"], "ckks_batch":fr["time_ckks_batch"]} for fr in fold_results])
print("\nAverage runtimes (s):", times.mean().to_dict())

Device: cpu
Loading Cora dataset...
Cora loaded — nodes: 2708, features: 1433, classes: 7

=== Fold 1/5 ===
train/val/test sizes (nodes): 1895 271 542
Epoch 001 train_loss: 5.5422 val_loss: 3.9894 wait: 0
Epoch 010 train_loss: 3.1065 val_loss: 2.7575 wait: 0
Epoch 020 train_loss: 2.7636 val_loss: 1.7463 wait: 0
Epoch 030 train_loss: 2.7124 val_loss: 1.0703 wait: 0
Epoch 040 train_loss: 2.6781 val_loss: 0.7710 wait: 0
Epoch 050 train_loss: 2.6965 val_loss: 0.6628 wait: 0
Epoch 060 train_loss: 2.6581 val_loss: 0.6277 wait: 2
Epoch 070 train_loss: 2.5558 val_loss: 0.6167 wait: 0
Epoch 080 train_loss: 2.6002 val_loss: 0.5931 wait: 5
Epoch 090 train_loss: 2.6677 val_loss: 0.5753 wait: 0
Epoch 100 train_loss: 2.5579 val_loss: 0.5662 wait: 0
Epoch 110 train_loss: 2.5475 val_loss: 0.5616 wait: 3
Epoch 120 train_loss: 2.5467 val_loss: 0.5250 wait: 0
Epoch 130 train_loss: 2.5584 val_loss: 0.5285 wait: 6
Epoch 140 train_loss: 2.5453 val_loss: 0.5183 wait: 16
Early stopping at epoch 144 (wait=20 >

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1256: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. Use OneVsRestClassifier(LogisticRegression(..)) instead. Leave it to its default value to avoid this warning.
  warnings.warn(


Fold results (Plain / Paillier / CKKS single / CKKS batched):
Plain: {'accuracy': 0.8634686346863468, 'precision_macro': 0.8567458936106032, 'recall_macro': 0.8616034956240011, 'f1_macro': 0.8587963096349917, 'roc_auc_ovr': 0.9761532013949247, 'log_loss': 0.5729238146225879}
Paillier: {'accuracy': 0.8634686346863468, 'precision_macro': 0.8567458936106032, 'recall_macro': 0.8616034956240011, 'f1_macro': 0.8587963096349917, 'roc_auc_ovr': 0.9764259230948306, 'log_loss': 1.1718724423958629}
CKKS single: {'accuracy': 0.8634686346863468, 'precision_macro': 0.8567458936106032, 'recall_macro': 0.8616034956240011, 'f1_macro': 0.8587963096349917, 'roc_auc_ovr': 0.9764086578542097, 'log_loss': 1.1898103091144927}
CKKS batched: {'accuracy': 0.8634686346863468, 'precision_macro': 0.8567458936106032, 'recall_macro': 0.8616034956240011, 'f1_macro': 0.8587963096349917, 'roc_auc_ovr': 0.9764587459872045, 'log_loss': 1.1906841631786806}
Runtimes (s): paillier=853.29, ckks_single=25.34, ckks_batch=25.33

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1256: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. Use OneVsRestClassifier(LogisticRegression(..)) instead. Leave it to its default value to avoid this warning.
  warnings.warn(


Fold results (Plain / Paillier / CKKS single / CKKS batched):
Plain: {'accuracy': 0.8653136531365314, 'precision_macro': 0.8541712564768283, 'recall_macro': 0.8505731579831781, 'f1_macro': 0.851404129532261, 'roc_auc_ovr': 0.9778668391137825, 'log_loss': 0.5428738532381637}
Paillier: {'accuracy': 0.8653136531365314, 'precision_macro': 0.8541712564768283, 'recall_macro': 0.8505731579831781, 'f1_macro': 0.851404129532261, 'roc_auc_ovr': 0.9779230206264835, 'log_loss': 1.0793986820456811}
CKKS single: {'accuracy': 0.8653136531365314, 'precision_macro': 0.8541712564768283, 'recall_macro': 0.8505731579831781, 'f1_macro': 0.851404129532261, 'roc_auc_ovr': 0.9778865942048852, 'log_loss': 1.0955981245616389}
CKKS batched: {'accuracy': 0.8653136531365314, 'precision_macro': 0.8541712564768283, 'recall_macro': 0.8505731579831781, 'f1_macro': 0.851404129532261, 'roc_auc_ovr': 0.9778620770950204, 'log_loss': 1.095973959490932}
Runtimes (s): paillier=859.09, ckks_single=25.49, ckks_batch=25.44

===

...

Fold results (Plain / Paillier / CKKS single / CKKS batched):
Plain: {'accuracy': 0.8671586715867159, 'precision_macro': 0.8536273167508927, 'recall_macro': 0.8508445314219467, 'f1_macro': 0.850758157713234, 'roc_auc_ovr': 0.9812824737717373, 'log_loss': 0.46916260387303443}
Paillier: {'accuracy': 0.8671586715867159, 'precision_macro': 0.8536273167508927, 'recall_macro': 0.8508445314219467, 'f1_macro': 0.850758157713234, 'roc_auc_ovr': 0.9824551774566436, 'log_loss': 0.8340154223139804}
CKKS single: {'accuracy': 0.8671586715867159, 'precision_macro': 0.8536273167508927, 'recall_macro': 0.8508445314219467, 'f1_macro': 0.850758157713234, 'roc_auc_ovr': 0.9824055096511143, 'log_loss': 0.8459674907773844}
CKKS batched: {'accuracy': 0.8671586715867159, 'precision_macro': 0.8536273167508927, 'recall_macro': 0.8508445314219467, 'f1_macro': 0.850758157713234, 'roc_auc_ovr': 0.9823506920502357, 'log_loss': 0.8467525292135575}
Runtimes (s): paillier=869.30, ckks_single=25.52, ckks_batch=25.54

=

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1256: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. Use OneVsRestClassifier(LogisticRegression(..)) instead. Leave it to its default value to avoid this warning.
  warnings.warn(


Fold results (Plain / Paillier / CKKS single / CKKS batched):
Plain: {'accuracy': 0.8687615526802218, 'precision_macro': 0.8589683980363422, 'recall_macro': 0.8651486654187642, 'f1_macro': 0.8609712772371981, 'roc_auc_ovr': 0.9765947981613777, 'log_loss': 0.5307905958865773}
Paillier: {'accuracy': 0.8687615526802218, 'precision_macro': 0.8589683980363422, 'recall_macro': 0.8651486654187642, 'f1_macro': 0.8609712772371981, 'roc_auc_ovr': 0.9796559786923174, 'log_loss': 0.9422765757227854}
CKKS single: {'accuracy': 0.866913123844732, 'precision_macro': 0.8573073653150204, 'recall_macro': 0.8642722412294566, 'f1_macro': 0.8595405071559891, 'roc_auc_ovr': 0.9795737285891172, 'log_loss': 0.9574683202049944}
CKKS batched: {'accuracy': 0.866913123844732, 'precision_macro': 0.8573073653150204, 'recall_macro': 0.8642722412294566, 'f1_macro': 0.8595405071559891, 'roc_auc_ovr': 0.9796412232566605, 'log_loss': 0.9581887229874437}
Runtimes (s): paillier=855.81, ckks_single=25.57, ckks_batch=25.78



/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1256: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. Use OneVsRestClassifier(LogisticRegression(..)) instead. Leave it to its default value to avoid this warning.
  warnings.warn(


Fold results (Plain / Paillier / CKKS single / CKKS batched):
Plain: {'accuracy': 0.88909426987061, 'precision_macro': 0.8768348204468248, 'recall_macro': 0.888637591658089, 'f1_macro': 0.8817174580550444, 'roc_auc_ovr': 0.9823261909471187, 'log_loss': 0.4707285900363585}
Paillier: {'accuracy': 0.88909426987061, 'precision_macro': 0.8768348204468248, 'recall_macro': 0.888637591658089, 'f1_macro': 0.8817174580550444, 'roc_auc_ovr': 0.9818515456266519, 'log_loss': 0.8816730947660355}
CKKS single: {'accuracy': 0.88909426987061, 'precision_macro': 0.8768348204468248, 'recall_macro': 0.888637591658089, 'f1_macro': 0.8817174580550444, 'roc_auc_ovr': 0.9818417267114422, 'log_loss': 0.8955355441664244}
CKKS batched: {'accuracy': 0.88909426987061, 'precision_macro': 0.8768348204468248, 'recall_macro': 0.888637591658089, 'f1_macro': 0.8817174580550444, 'roc_auc_ovr': 0.9818576056223106, 'log_loss': 0.894788384050891}
Runtimes (s): paillier=863.19, ckks_single=25.53, ckks_batch=25.40

=== Aggrega

# **GraphSAGE Model**

In [ ]:
# =============================================================================
# Faster Cora node-classification pipeline with PCA + smaller HE params
# GraphSAGE -> PCA (dim reduction) -> LogisticRegression (OvR) -> HE inference (Paillier & CKKS)
# Notes:
#  - HE inference runs OvR: one encrypted dot per class per test sample.
#  - PCA reduces dimension to speed up HE drastically.
#  - Paillier key default 1024 bits; CKKS poly 4096 with coeffs [40,20,40].
# =============================================================================

import time, random, math
import numpy as np
import torch
import torch.nn.functional as F
from copy import deepcopy
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, log_loss
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from torch_geometric.datasets import Planetoid
from torch_geometric.nn import SAGEConv
from torch.nn import BatchNorm1d
import torch.nn as nn

import phe as paillier
import tenseal as ts
import pandas as pd
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

# ---------------- USER KNOBS (tune these for speed vs accuracy) ----------------
RND = 42
N_FOLDS = 5

# GraphSAGE params (node-level)
SAGE_HIDDEN = 128
SAGE_EMB = 256
SAGE_EPOCHS = 40        # reduced epochs for speed (increase if you want better embeddings)
SAGE_PATIENCE = 8

# PCA dimension (important: smaller -> faster HE)
PCA_DIM = 64            # recommended: 32..128; set lower for big speedups

# Paillier tuning (smaller -> faster)
PAILLIER_SCALE = 1000
PAILLIER_KEY = 1024     # 1024-bit for speed (use 2048 for stronger security)

# CKKS tuning (smaller poly -> faster)
CKKS_POLY = 4096
CKKS_CTX_SCALE_EXP = 20
CKKS_BATCH_SIZE = 32
CKKS_COEFFS = [40, 20, 40]

# LR / augmentation
LR_AUGMENT_COPIES = 1
LR_AUG_NOISE_STD = 5e-3

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# ------------------------------------------------------------------------------

# reproducibility
random.seed(RND)
np.random.seed(RND)
torch.manual_seed(RND)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RND)
device = torch.device(DEVICE)
print("Device:", device)

# ---------------- GraphSAGE (node-level) ----------------
class SAGENodeModel(nn.Module):
    def __init__(self, in_ch, hidden=SAGE_HIDDEN, emb_dim=SAGE_EMB, dropout=0.4):
        super().__init__()
        self.conv1 = SAGEConv(in_ch, hidden)
        self.bn1 = BatchNorm1d(hidden)
        self.conv2 = SAGEConv(hidden, hidden)
        self.bn2 = BatchNorm1d(hidden)
        self.conv3 = SAGEConv(hidden, hidden)
        self.bn3 = BatchNorm1d(hidden)
        self.lin = nn.Linear(hidden, emb_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, edge_index):
        x = F.relu(self.bn1(self.conv1(x, edge_index)))
        x = F.relu(self.bn2(self.conv2(x, edge_index)))
        x = F.relu(self.bn3(self.conv3(x, edge_index)))
        x = F.relu(self.lin(x))
        x = self.dropout(x)
        return x  # [num_nodes, emb_dim]

# ---------------- training helper (node-level) ----------------
def train_sage_node(data, train_idx, val_idx, epochs=SAGE_EPOCHS, patience=SAGE_PATIENCE):
    model = SAGENodeModel(data.num_features).to(device)
    head = torch.nn.Linear(SAGE_EMB, int(data.y.max().item() + 1)).to(device)
    opt = torch.optim.AdamW(list(model.parameters()) + list(head.parameters()), lr=5e-4, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    crit = nn.CrossEntropyLoss()

    best_val = float("inf"); wait = 0; best_state = None
    for ep in range(1, epochs+1):
        model.train(); head.train()
        opt.zero_grad()
        emb = model(data.x.to(device), data.edge_index.to(device))
        out = head(emb)
        loss = crit(out[train_idx], data.y[train_idx].to(device))
        loss.backward(); opt.step()
        sched.step()

        model.eval()
        with torch.no_grad():
            emb_val = model(data.x.to(device), data.edge_index.to(device))
            out_val = head(emb_val)
            vloss = crit(out_val[val_idx], data.y[val_idx].to(device)).item()

        if vloss < best_val - 1e-6:
            best_val = vloss; wait = 0
            best_state = (deepcopy(model.state_dict()), deepcopy(head.state_dict()))
        else:
            wait += 1

        if ep % 5 == 0 or ep == 1:
            print(f"Epoch {ep:03d} train_loss: {loss.item():.4f} val_loss: {vloss:.4f} wait: {wait}")

        if wait >= patience:
            print("Early stopping at epoch", ep)
            break

    if best_state is not None:
        model.load_state_dict(best_state[0]); head.load_state_dict(best_state[1])
    return model

# ---------------- metrics (multiclass) ----------------
def softmax_rows(logits):
    l = np.array(logits, dtype=float)
    l = l - np.max(l, axis=1, keepdims=True)
    ex = np.exp(l)
    return ex / np.sum(ex, axis=1, keepdims=True)

def metrics_from_probs_multiclass(probs, y_true):
    preds = np.argmax(probs, axis=1)
    try:
        roc = roc_auc_score(y_true, probs, multi_class='ovr')
    except Exception:
        roc = float("nan")
    return {
        "accuracy": float(accuracy_score(y_true, preds)),
        "precision_macro": float(precision_score(y_true, preds, average='macro', zero_division=0)),
        "recall_macro": float(recall_score(y_true, preds, average='macro', zero_division=0)),
        "f1_macro": float(f1_score(y_true, preds, average='macro', zero_division=0)),
        "roc_auc_ovr": float(roc) if not np.isnan(roc) else float("nan"),
        "log_loss": float(log_loss(y_true, probs))
    }

# ---------------- Paillier multiclass (OvR) optimized for reduced dim ----------------
def paillier_inference_multiclass(X, W, b, scale=PAILLIER_SCALE, key_size=PAILLIER_KEY):
    """
    X: [n_samples, n_features_reduced]
    W: [n_classes, n_features_reduced]
    b: [n_classes]
    Note: encrypts per-feature-per-sample -> keep n_features small (PCA).
    """
    t0_all = time.time()
    pk, sk = paillier.generate_paillier_keypair(n_length=key_size)
    W_int = [[int(round(float(wij) * scale)) for wij in w_row] for w_row in W.tolist()]
    b_int = [int(round(float(bc) * (scale ** 2))) for bc in b.tolist()]

    logits = []
    t0 = time.time()
    for x in X:
        x_int = [int(round(float(xj) * scale)) for xj in x.tolist()]
        enc_x = [pk.encrypt(int(xj)) for xj in x_int]
        class_scores = []
        for class_idx, w_int_row in enumerate(W_int):
            acc = pk.encrypt(int(b_int[class_idx]))
            for enc_feat, wi in zip(enc_x, w_int_row):
                acc = acc + (enc_feat * int(wi))
            dec_int = sk.decrypt(acc)
            score = float(dec_int) / (scale ** 2)
            class_scores.append(score)
        logits.append(class_scores)
    wall = time.time() - t0
    wall_all = time.time() - t0_all
    return np.array(logits), wall, wall_all

# ---------------- CKKS multiclass (single & batched) ----------------
def ckks_single_inference_multiclass(X, W, b, poly_mod=CKKS_POLY, ctx_scale_exp=CKKS_CTX_SCALE_EXP, coeff_mod_bit_sizes=CKKS_COEFFS):
    t0_all = time.time()
    ctx = ts.context(ts.SCHEME_TYPE.CKKS, poly_modulus_degree=poly_mod, coeff_mod_bit_sizes=coeff_mod_bit_sizes)
    ctx.global_scale = 2 ** ctx_scale_exp
    ctx.generate_galois_keys()
    enc_w_list = [ts.ckks_vector(ctx, w_row.tolist()) for w_row in W]
    logits = []
    t0 = time.time()
    for x in X:
        enc_x = ts.ckks_vector(ctx, x.tolist())
        class_scores = []
        for enc_w in enc_w_list:
            enc_dot = enc_x.dot(enc_w)
            dec_dot = enc_dot.decrypt()
            val = float(dec_dot[0]) if hasattr(dec_dot, "__len__") else float(dec_dot)
            class_scores.append(val)
        class_scores = np.array(class_scores) + b
        logits.append(class_scores.tolist())
    wall = time.time() - t0
    wall_all = time.time() - t0_all
    return np.array(logits), wall, wall_all

def ckks_batched_inference_multiclass(X, W, b, batch_size=CKKS_BATCH_SIZE, poly_mod=CKKS_POLY, ctx_scale_exp=CKKS_CTX_SCALE_EXP, coeff_mod_bit_sizes=CKKS_COEFFS):
    t0_all = time.time()
    ctx = ts.context(ts.SCHEME_TYPE.CKKS, poly_modulus_degree=poly_mod, coeff_mod_bit_sizes=coeff_mod_bit_sizes)
    ctx.global_scale = 2 ** ctx_scale_exp
    ctx.generate_galois_keys()
    enc_w_list = [ts.ckks_vector(ctx, w_row.tolist()) for w_row in W]
    n = X.shape[0]
    logits = []
    t0 = time.time()
    for start in range(0, n, batch_size):
        end = min(start + batch_size, n)
        batch = X[start:end]
        # encrypt each sample in batch and compute per-class dot
        for x in batch:
            enc_x = ts.ckks_vector(ctx, x.tolist())
            class_scores = []
            for enc_w in enc_w_list:
                enc_dot = enc_x.dot(enc_w)
                dec_dot = enc_dot.decrypt()
                val = float(dec_dot[0]) if hasattr(dec_dot, "__len__") else float(dec_dot)
                class_scores.append(val)
            class_scores = np.array(class_scores) + b
            logits.append(class_scores.tolist())
    wall = time.time() - t0
    wall_all = time.time() - t0_all
    return np.array(logits), wall, wall_all

# ---------------- main CV loop (Cora node-level) ----------------
print("Loading Cora dataset...")
dataset = Planetoid(root="/tmp/Cora_pca_he", name="Cora")
data = dataset[0]
n_nodes = data.num_nodes
labels = data.y.cpu().numpy()
num_classes = dataset.num_classes
print(f"Cora: nodes={n_nodes}, features={data.num_features}, classes={num_classes}")

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RND)
fold_results = []
fold = 0

for trainval_idx, test_idx in skf.split(np.arange(n_nodes), labels):
    fold += 1
    print(f"\n=== Fold {fold}/{N_FOLDS} ===")
    # split train/val
    train_idx, val_idx = train_test_split(trainval_idx, test_size=0.125, random_state=RND, stratify=labels[trainval_idx])
    train_idx_t = torch.tensor(train_idx, dtype=torch.long)
    val_idx_t = torch.tensor(val_idx, dtype=torch.long)
    test_idx_t = torch.tensor(test_idx, dtype=torch.long)

    # 1) train GraphSAGE node-level
    t0 = time.time()
    sage = train_sage_node(data, train_idx_t, val_idx_t, epochs=SAGE_EPOCHS, patience=SAGE_PATIENCE)
    t_sage = time.time() - t0
    print(f"GraphSAGE training time: {t_sage:.2f}s")

    # 2) extract node embeddings for all nodes
    sage.eval()
    with torch.no_grad():
        emb_all = sage(data.x.to(device), data.edge_index.to(device)).cpu().numpy()  # [n_nodes, emb_dim]

    # 3) PCA on embeddings (fit on train+val embeddings)
    X_train_emb = emb_all[train_idx]
    X_val_emb = emb_all[val_idx]
    X_test_emb = emb_all[test_idx]
    print("Embedding shapes (train/val/test):", X_train_emb.shape, X_val_emb.shape, X_test_emb.shape)

    scaler_emb = StandardScaler()
    scaler_emb.fit(np.vstack([X_train_emb, X_val_emb]))
    X_train_scaled = scaler_emb.transform(X_train_emb)
    X_val_scaled = scaler_emb.transform(X_val_emb)
    X_test_scaled = scaler_emb.transform(X_test_emb)

    # apply PCA to reduce embedding dim
    pca = PCA(n_components=min(PCA_DIM, X_train_scaled.shape[1]), random_state=RND)
    pca.fit(np.vstack([X_train_scaled, X_val_scaled]))
    X_train_p = pca.transform(X_train_scaled)
    X_val_p = pca.transform(X_val_scaled)
    X_test_p = pca.transform(X_test_scaled)
    print("PCA reduced dim ->", X_train_p.shape[1])

    # 4) augment and train Logistic Regression (OvR)
    def augment(X, y, copies=LR_AUGMENT_COPIES, noise_std=LR_AUG_NOISE_STD):
        Xs, ys = [X], [y]
        for _ in range(copies):
            Xs.append(X + np.random.normal(scale=noise_std * np.std(X, axis=0, ddof=0), size=X.shape))
            ys.append(y.copy())
        return np.vstack(Xs), np.concatenate(ys)

    X_aug, y_aug = augment(np.vstack([X_train_p, X_val_p]), np.concatenate([labels[train_idx], labels[val_idx]]))
    clf = LogisticRegression(max_iter=10000, solver="lbfgs", class_weight="balanced", multi_class="ovr")
    t0 = time.time()
    clf.fit(X_aug, y_aug)
    t_lr = time.time() - t0
    print(f"LogisticRegression training time: {t_lr:.2f}s")

    # 5) plaintext evaluation
    probs_plain = clf.predict_proba(X_test_p)
    metrics_plain = metrics_from_probs_multiclass(probs_plain, labels[test_idx])
    print("Plaintext metrics:", metrics_plain)

    # 6) prepare W and b (reduced-dim weights)
    W = clf.coef_.astype(float)    # shape [n_classes, PCA_DIM]
    b = clf.intercept_.astype(float)
    if W.ndim == 1:
        W = W.reshape(1, -1)

    # 7) Paillier multiclass (OvR) on test set only
    print("Running Paillier OvR on test set (this may still take time depending on PCA_DIM & test size)...")
    t0 = time.time()
    paillier_logits, paillier_time_inner, paillier_time_total = paillier_inference_multiclass(X_test_p, W, b, scale=PAILLIER_SCALE, key_size=PAILLIER_KEY)
    t_paillier = time.time() - t0
    print(f"Paillier microtime: {paillier_time_inner:.2f}s ; total paillier function wall: {paillier_time_total:.2f}s ; measured: {t_paillier:.2f}s")
    probs_paillier = softmax_rows(paillier_logits)
    metrics_paillier = metrics_from_probs_multiclass(probs_paillier, labels[test_idx])
    print("Paillier metrics:", metrics_paillier)

    # 8) CKKS single-sample multiclass
    print("Running CKKS single-sample OvR on test set...")
    t0 = time.time()
    ckks_single_logits, ckks_single_inner, ckks_single_total = ckks_single_inference_multiclass(X_test_p, W, b, poly_mod=CKKS_POLY, ctx_scale_exp=CKKS_CTX_SCALE_EXP, coeff_mod_bit_sizes=CKKS_COEFFS)
    t_ckks_single = time.time() - t0
    print(f"CKKS single inner: {ckks_single_inner:.2f}s ; total: {ckks_single_total:.2f}s ; measured: {t_ckks_single:.2f}s")
    probs_ckks_single = softmax_rows(ckks_single_logits)
    metrics_ckks_single = metrics_from_probs_multiclass(probs_ckks_single, labels[test_idx])
    print("CKKS single metrics:", metrics_ckks_single)

    # 9) CKKS batched multiclass
    print("Running CKKS batched OvR on test set...")
    t0 = time.time()
    ckks_batch_logits, ckks_batch_inner, ckks_batch_total = ckks_batched_inference_multiclass(X_test_p, W, b, batch_size=CKKS_BATCH_SIZE, poly_mod=CKKS_POLY, ctx_scale_exp=CKKS_CTX_SCALE_EXP, coeff_mod_bit_sizes=CKKS_COEFFS)
    t_ckks_batch = time.time() - t0
    print(f"CKKS batch inner: {ckks_batch_inner:.2f}s ; total: {ckks_batch_total:.2f}s ; measured: {t_ckks_batch:.2f}s")
    probs_ckks_batch = softmax_rows(ckks_batch_logits)
    metrics_ckks_batch = metrics_from_probs_multiclass(probs_ckks_batch, labels[test_idx])
    print("CKKS batch metrics:", metrics_ckks_batch)

    fold_results.append({
        "plain": metrics_plain, "paillier": metrics_paillier,
        "ckks_single": metrics_ckks_single, "ckks_batch": metrics_ckks_batch,
        "time_graphsage": t_sage, "time_lr": t_lr,
        "time_paillier": paillier_time_total, "time_ckks_single": ckks_single_total, "time_ckks_batch": ckks_batch_total
    })

# ---------- aggregate ----------
import pandas as pd

def agg(key):
    arr = [fr[key] for fr in fold_results]
    df = pd.DataFrame(arr)
    mean = df.mean().to_dict(); std = df.std().to_dict()
    return {k: f"{mean[k]:.4f} ± {std[k]:.4f}" for k in mean.keys()}

print("\n=== Aggregated (mean ± std) ===")
print("Plaintext:", agg("plain"))
print("Paillier :", agg("paillier"))
print("CKKS sing:", agg("ckks_single"))
print("CKKS bat :", agg("ckks_batch"))

times = pd.DataFrame([{"paillier":fr["time_paillier"], "ckks_single":fr["time_ckks_single"], "ckks_batch":fr["time_ckks_batch"]} for fr in fold_results])
print("\nAverage runtimes (s):", times.mean().to_dict())


Device: cpu
Loading Cora dataset...
Cora: nodes=2708, features=1433, classes=7

=== Fold 1/5 ===
Epoch 001 train_loss: 1.9607 val_loss: 1.9460 wait: 0
Epoch 005 train_loss: 1.5903 val_loss: 1.9228 wait: 0
Epoch 010 train_loss: 1.2410 val_loss: 1.8682 wait: 0
Epoch 015 train_loss: 0.9627 val_loss: 1.7709 wait: 0
Epoch 020 train_loss: 0.7705 val_loss: 1.6358 wait: 0
Epoch 025 train_loss: 0.6440 val_loss: 1.4777 wait: 0
Epoch 030 train_loss: 0.5735 val_loss: 1.3122 wait: 0
Epoch 035 train_loss: 0.5376 val_loss: 1.1575 wait: 0
Epoch 040 train_loss: 0.5330 val_loss: 1.0259 wait: 0
GraphSAGE training time: 19.11s
Embedding shapes (train/val/test): (1895, 256) (271, 256) (542, 256)
PCA reduced dim -> 64


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1256: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. Use OneVsRestClassifier(LogisticRegression(..)) instead. Leave it to its default value to avoid this warning.
  warnings.warn(


LogisticRegression training time: 0.43s
Plaintext metrics: {'accuracy': 0.8394833948339483, 'precision_macro': 0.8192387896038478, 'recall_macro': 0.8349486514212414, 'f1_macro': 0.8261007432550374, 'roc_auc_ovr': 0.9701598784749862, 'log_loss': 0.7015123423892188}
Running Paillier OvR on test set (this may still take time depending on PCA_DIM & test size)...
Paillier microtime: 851.84s ; total paillier function wall: 852.01s ; measured: 852.01s
Paillier metrics: {'accuracy': 0.8394833948339483, 'precision_macro': 0.8192387896038478, 'recall_macro': 0.8349486514212414, 'f1_macro': 0.8261007432550374, 'roc_auc_ovr': 0.9720626197499611, 'log_loss': 1.1449714782588367}
Running CKKS single-sample OvR on test set...
CKKS single inner: 24.75s ; total: 24.84s ; measured: 24.84s
CKKS single metrics: {'accuracy': 0.8394833948339483, 'precision_macro': 0.8192387896038478, 'recall_macro': 0.8349486514212414, 'f1_macro': 0.8261007432550374, 'roc_auc_ovr': 0.9720888064948342, 'log_loss': 1.16485311

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1256: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. Use OneVsRestClassifier(LogisticRegression(..)) instead. Leave it to its default value to avoid this warning.
  warnings.warn(


LogisticRegression training time: 0.37s
Plaintext metrics: {'accuracy': 0.8616236162361623, 'precision_macro': 0.8393337176850031, 'recall_macro': 0.8605017995097454, 'f1_macro': 0.8481779677865816, 'roc_auc_ovr': 0.974411869871796, 'log_loss': 0.61580788664197}
Running Paillier OvR on test set (this may still take time depending on PCA_DIM & test size)...
Paillier microtime: 853.43s ; total paillier function wall: 853.55s ; measured: 853.55s
Paillier metrics: {'accuracy': 0.8616236162361623, 'precision_macro': 0.8393337176850031, 'recall_macro': 0.8605017995097454, 'f1_macro': 0.8481779677865816, 'roc_auc_ovr': 0.974286212638859, 'log_loss': 1.0169323532982537}
Running CKKS single-sample OvR on test set...
CKKS single inner: 24.48s ; total: 24.57s ; measured: 24.57s
CKKS single metrics: {'accuracy': 0.8616236162361623, 'precision_macro': 0.8384876242883329, 'recall_macro': 0.8605017995097454, 'f1_macro': 0.8474836810729656, 'roc_auc_ovr': 0.97428929512725, 'log_loss': 1.03645189934730

...

LogisticRegression training time: 0.38s
Plaintext metrics: {'accuracy': 0.8542435424354243, 'precision_macro': 0.8358594457638331, 'recall_macro': 0.8457630849655045, 'f1_macro': 0.8390722768638936, 'roc_auc_ovr': 0.9761925793004369, 'log_loss': 0.5717894221909002}
Running Paillier OvR on test set (this may still take time depending on PCA_DIM & test size)...
Paillier microtime: 869.22s ; total paillier function wall: 869.54s ; measured: 869.54s
Paillier metrics: {'accuracy': 0.8542435424354243, 'precision_macro': 0.8358594457638331, 'recall_macro': 0.8457630849655045, 'f1_macro': 0.8390722768638936, 'roc_auc_ovr': 0.9772477319013628, 'log_loss': 0.8630708247364387}
Running CKKS single-sample OvR on test set...
CKKS single inner: 24.89s ; total: 24.97s ; measured: 24.97s
CKKS single metrics: {'accuracy': 0.8542435424354243, 'precision_macro': 0.8358594457638331, 'recall_macro': 0.8457630849655045, 'f1_macro': 0.8390722768638936, 'roc_auc_ovr': 0.9772050761716421, 'log_loss': 0.87824566

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1256: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. Use OneVsRestClassifier(LogisticRegression(..)) instead. Leave it to its default value to avoid this warning.
  warnings.warn(


LogisticRegression training time: 0.36s
Plaintext metrics: {'accuracy': 0.8299445471349353, 'precision_macro': 0.813528300973981, 'recall_macro': 0.8220913311751916, 'f1_macro': 0.8147865308553214, 'roc_auc_ovr': 0.9677333384653097, 'log_loss': 0.757627169607833}
Running Paillier OvR on test set (this may still take time depending on PCA_DIM & test size)...
Paillier microtime: 844.60s ; total paillier function wall: 844.72s ; measured: 844.72s
Paillier metrics: {'accuracy': 0.8299445471349353, 'precision_macro': 0.813528300973981, 'recall_macro': 0.8220913311751916, 'f1_macro': 0.8147865308553214, 'roc_auc_ovr': 0.9701785528115898, 'log_loss': 1.2009112492855043}
Running CKKS single-sample OvR on test set...
CKKS single inner: 24.55s ; total: 24.64s ; measured: 24.64s
CKKS single metrics: {'accuracy': 0.8280961182994455, 'precision_macro': 0.8112731626764057, 'recall_macro': 0.8212149069858841, 'f1_macro': 0.812845391209566, 'roc_auc_ovr': 0.9700646037090973, 'log_loss': 1.221779619826

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1256: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. Use OneVsRestClassifier(LogisticRegression(..)) instead. Leave it to its default value to avoid this warning.
  warnings.warn(


LogisticRegression training time: 0.36s
Plaintext metrics: {'accuracy': 0.8502772643253235, 'precision_macro': 0.8295188825450952, 'recall_macro': 0.8540500824440375, 'f1_macro': 0.8396499410886039, 'roc_auc_ovr': 0.9763104638875749, 'log_loss': 0.6213728134669331}
Running Paillier OvR on test set (this may still take time depending on PCA_DIM & test size)...
Paillier microtime: 855.37s ; total paillier function wall: 855.47s ; measured: 855.47s
Paillier metrics: {'accuracy': 0.8502772643253235, 'precision_macro': 0.8295188825450952, 'recall_macro': 0.8540500824440375, 'f1_macro': 0.8396499410886039, 'roc_auc_ovr': 0.9782218327206395, 'log_loss': 0.9788607044249376}
Running CKKS single-sample OvR on test set...
CKKS single inner: 24.54s ; total: 24.65s ; measured: 24.65s
CKKS single metrics: {'accuracy': 0.8484288354898336, 'precision_macro': 0.8281632706530182, 'recall_macro': 0.8531736582547299, 'f1_macro': 0.8384251315578901, 'roc_auc_ovr': 0.9781868889160973, 'log_loss': 0.99678598

# **Proposed HybridGNN Cora**

In [7]:
# =============================================================================
# Hybrid GAT+GIN++ pipeline with improvements:
# - deeper hybrid encoder (residuals + layernorm + gated fusion)
# - DropEdge, label smoothing, AdamW + CosineAnnealingLR
# - PCA before HE (optional) to reduce HE cost
# - CKKS scale increased to 2**40
# - Real Paillier + CKKS (single + batched)
# - Aggregated results printed as "mean ± std" dicts
# Dataset: Cora (Planetoid)
# =============================================================================

import os, time, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, log_loss
from torch_geometric.datasets import Planetoid
from torch_geometric.nn import GATConv, GINConv
from torch_geometric.utils import dropout_edge
from torch.nn import LayerNorm
import phe as paillier
import tenseal as ts
from tqdm import tqdm
import pandas as pd

# ---------------- USER KNOBS ----------------
RND = 42
N_FOLDS = 3
EPOCHS = 120
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# encoder params
HIDDEN = 128
EMB_DIM = 256
HEADS = 8
DROPOUT = 0.6
ATTN_DROPOUT = 0.3

# training
LR = 2e-3
WEIGHT_DECAY = 1e-4

# label smoothing
LABEL_SMOOTH = 0.1

# DropEdge
DROPEDGE_P = 0.1

# PCA before HE
USE_PCA_BEFORE_HE = True
PCA_DIM = 64

# Paillier params
PAILLIER_SCALE = 5000
PAILLIER_KEY = 1024

# CKKS params
CKKS_POLY = 8192
CKKS_CTX_SCALE_EXP = 40
CKKS_BATCH_SIZE = 16
CKKS_COEFFS = [60, 40, 40, 60]

# ---------------------------------------------------------
random.seed(RND)
np.random.seed(RND)
torch.manual_seed(RND)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RND)
device = torch.device(DEVICE)
print("Device:", device)

# ---------------- Dataset ----------------
dataset = Planetoid(root="/tmp/Cora_improved", name="Cora")
data = dataset[0]
IN_DIM = data.num_features
NUM_CLASSES = dataset.num_classes
print(f"Cora: nodes={data.num_nodes}, features={IN_DIM}, classes={NUM_CLASSES}")

# ---------------- Utilities ----------------
def label_smooth_target(target, n_classes, eps):
    if eps <= 0:
        y = torch.zeros((target.size(0), n_classes), device=target.device)
        y[torch.arange(target.size(0)), target] = 1.0
        return y
    off_value = eps / (n_classes - 1)
    on_value = 1.0 - eps
    y = torch.full((target.size(0), n_classes), off_value, device=target.device)
    y[torch.arange(target.size(0)), target] = on_value
    return y

# ---------------- Hybrid GAT+GIN++ ----------------
class HybridGATGINpp(nn.Module):
    def __init__(self, in_ch, hidden=HIDDEN, emb_dim=EMB_DIM, heads=HEADS, dropout=DROPOUT, attn_dropout=ATTN_DROPOUT):
        super().__init__()
        self.gat1 = GATConv(in_ch, hidden, heads=heads, dropout=attn_dropout)
        self.gin1 = GINConv(nn.Sequential(nn.Linear(in_ch, hidden), nn.ReLU(), nn.Linear(hidden, hidden)))
        self.norm1 = LayerNorm(hidden * heads + hidden)
        self.res1 = nn.Linear(in_ch, hidden * heads + hidden)

        self.gat2 = GATConv(hidden * heads + hidden, hidden, heads=heads, dropout=attn_dropout)
        self.gin2 = GINConv(nn.Sequential(nn.Linear(hidden * heads + hidden, hidden), nn.ReLU(), nn.Linear(hidden, hidden)))
        self.norm2 = LayerNorm(hidden * heads + hidden)
        self.res2 = nn.Linear(hidden * heads + hidden, hidden * heads + hidden)

        self.gate1 = nn.Parameter(torch.tensor(0.5))
        self.fc_out = nn.Linear(hidden * heads + hidden, emb_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, edge_index, dropedge_p=0.0, training=False):
        if training and dropedge_p > 0:
            edge_index, _ = dropout_edge(edge_index, p=dropedge_p, force_undirected=True)

        gat1 = self.gat1(x, edge_index)
        gin1 = self.gin1(x, edge_index)
        cat1 = torch.cat([gat1, gin1], dim=-1)
        res = self.res1(x)
        h1 = F.relu(self.norm1(cat1 + res))
        h1 = self.dropout(h1)

        gat2 = self.gat2(h1, edge_index)
        gin2 = self.gin2(h1, edge_index)
        cat2 = torch.cat([gat2, gin2], dim=-1)
        res2 = self.res2(h1)
        h2 = F.relu(self.norm2(cat2 + res2))
        h2 = self.dropout(h2)

        g = torch.sigmoid(self.gate1)
        fused = g * h2 + (1.0 - g) * h1
        out = self.fc_out(fused)
        return out

# ---------------- Training helper (no early stopping) ----------------
def train_encoder(data, train_idx, val_idx, epochs=EPOCHS, dropedge_p=DROPEDGE_P):
    model = HybridGATGINpp(data.num_features).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(opt, T_0=20, T_mult=1)
    n_classes = int(data.y.max().item()) + 1

    for ep in range(1, epochs+1):
        model.train()
        opt.zero_grad()
        out = model(data.x.to(device), data.edge_index.to(device), dropedge_p, training=True)
        head = torch.nn.Linear(out.size(1), n_classes).to(device)
        if LABEL_SMOOTH > 0:
            y_smooth = label_smooth_target(data.y.to(device), n_classes, LABEL_SMOOTH)
            logits = head(out)
            loss = torch.mean(-torch.sum(y_smooth * F.log_softmax(logits, dim=1), dim=1))
        else:
            logits = head(out)
            loss = F.cross_entropy(logits, data.y.to(device))
        loss.backward()
        opt.step()
        sched.step(ep)

        if ep % 10 == 0 or ep == 1:
            print(f"Epoch {ep:03d} loss: {loss.item():.4f}")

    return model

# ---------------- Metrics ----------------
def softmax_rows(logits):
    l = np.array(logits, dtype=float)
    l = l - np.max(l, axis=1, keepdims=True)
    ex = np.exp(l)
    return ex / np.sum(ex, axis=1, keepdims=True)

def metrics_from_probs_multiclass(probs, y_true):
    preds = np.argmax(probs, axis=1)
    try:
        roc = roc_auc_score(y_true, probs, multi_class='ovr')
    except Exception:
        roc = float("nan")
    return {
        "accuracy": float(accuracy_score(y_true, preds)),
        "precision_macro": float(precision_score(y_true, preds, average='macro', zero_division=0)),
        "recall_macro": float(recall_score(y_true, preds, average='macro', zero_division=0)),
        "f1_macro": float(f1_score(y_true, preds, average='macro', zero_division=0)),
        "roc_auc_ovr": float(roc) if not np.isnan(roc) else float("nan"),
        "log_loss": float(log_loss(y_true, probs))
    }

# ---------------- Paillier inference ----------------
def paillier_inference_multiclass(X, W, b, scale=PAILLIER_SCALE, key_size=PAILLIER_KEY):
    pk, sk = paillier.generate_paillier_keypair(n_length=key_size)
    W_int = [[int(round(float(wij) * scale)) for wij in w_row] for w_row in W.tolist()]
    b_int = [int(round(float(bc) * (scale ** 2))) for bc in b.tolist()]
    logits = []
    t0 = time.time()
    for x in X:
        x_int = [int(round(float(xj) * scale)) for xj in x.tolist()]
        enc_x = [pk.encrypt(int(v)) for v in x_int]
        cls_scores = []
        for cls_idx, wrow in enumerate(W_int):
            acc = pk.encrypt(int(b_int[cls_idx]))
            for enc_feat, wij in zip(enc_x, wrow):
                acc = acc + (enc_feat * int(wij))
            dec_int = sk.decrypt(acc)
            score = float(dec_int) / (scale ** 2)
            cls_scores.append(score)
        logits.append(cls_scores)
    wall = time.time() - t0
    return np.array(logits), wall

# ---------------- CKKS single inference ----------------
def ckks_single_inference_multiclass(X, W, b, poly_mod=CKKS_POLY, ctx_scale_exp=CKKS_CTX_SCALE_EXP, coeff_mod_bit_sizes=CKKS_COEFFS):
    ctx = ts.context(ts.SCHEME_TYPE.CKKS, poly_modulus_degree=poly_mod, coeff_mod_bit_sizes=coeff_mod_bit_sizes)
    ctx.global_scale = 2 ** ctx_scale_exp
    ctx.generate_galois_keys()
    enc_w_list = [ts.ckks_vector(ctx, w_row.tolist()) for w_row in W]
    logits = []
    t0 = time.time()
    for x in X:
        enc_x = ts.ckks_vector(ctx, x.tolist())
        cls_scores = []
        for enc_w in enc_w_list:
            enc_dot = enc_x.dot(enc_w)
            val = float(enc_dot.decrypt()[0])
            cls_scores.append(val)
        cls_scores = np.array(cls_scores) + b
        logits.append(cls_scores.tolist())
    wall = time.time() - t0
    return np.array(logits), wall

# ---------------- CKKS batched inference ----------------
def ckks_batched_inference_multiclass(X, W, b, batch_size=CKKS_BATCH_SIZE, poly_mod=CKKS_POLY, ctx_scale_exp=CKKS_CTX_SCALE_EXP, coeff_mod_bit_sizes=CKKS_COEFFS):
    ctx = ts.context(ts.SCHEME_TYPE.CKKS, poly_modulus_degree=poly_mod, coeff_mod_bit_sizes=coeff_mod_bit_sizes)
    ctx.global_scale = 2 ** ctx_scale_exp
    ctx.generate_galois_keys()
    enc_w_list = [ts.ckks_vector(ctx, w_row.tolist()) for w_row in W]
    n = X.shape[0]
    logits = []
    t0 = time.time()
    for start in range(0, n, batch_size):
        end = min(start + batch_size, n)
        batch = X[start:end]
        for x in batch:
            enc_x = ts.ckks_vector(ctx, x.tolist())
            cls_scores = []
            for enc_w in enc_w_list:
                enc_dot = enc_x.dot(enc_w)
                val = float(enc_dot.decrypt()[0])
                cls_scores.append(val)
            cls_scores = np.array(cls_scores) + b
            logits.append(cls_scores.tolist())
    wall = time.time() - t0
    return np.array(logits), wall

# ---------------- Main CV loop ----------------
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RND)
fold_results = []

for fold, (trainval_idx, test_idx) in enumerate(skf.split(np.arange(data.num_nodes), data.y.numpy()), 1):
    print(f"\n=== Fold {fold}/{N_FOLDS} ===")
    train_idx, val_idx = train_test_split(trainval_idx, test_size=0.125, random_state=RND, stratify=data.y.numpy()[trainval_idx])
    train_idx_t = torch.tensor(train_idx, dtype=torch.long)
    val_idx_t = torch.tensor(val_idx, dtype=torch.long)
    test_idx_t = torch.tensor(test_idx, dtype=torch.long)

    encoder = train_encoder(data, train_idx_t, val_idx_t, epochs=EPOCHS, dropedge_p=DROPEDGE_P)

    encoder.eval()
    with torch.no_grad():
        emb_all = encoder(data.x.to(device), data.edge_index.to(device), dropedge_p=0.0, training=False).cpu().numpy()
    X_train, y_train = emb_all[train_idx], data.y.numpy()[train_idx]
    X_val, y_val = emb_all[val_idx], data.y.numpy()[val_idx]
    X_test, y_test = emb_all[test_idx], data.y.numpy()[test_idx]

    scaler = StandardScaler()
    scaler.fit(np.vstack([X_train, X_val]))
    X_train_s = scaler.transform(X_train)
    X_val_s = scaler.transform(X_val)
    X_test_s = scaler.transform(X_test)

    if USE_PCA_BEFORE_HE:
        pca = PCA(n_components=min(PCA_DIM, X_train_s.shape[1]))
        pca.fit(np.vstack([X_train_s, X_val_s]))
        X_train_he = pca.transform(X_train_s)
        X_val_he = pca.transform(X_val_s)
        X_test_he = pca.transform(X_test_s)
    else:
        X_train_he, X_val_he, X_test_he = X_train_s, X_val_s, X_test_s

    clf = LogisticRegression(max_iter=5000, solver="lbfgs", class_weight="balanced", multi_class="ovr")
    clf.fit(np.vstack([X_train_s, X_val_s]), np.concatenate([y_train, y_val]))

    probs_plain = clf.predict_proba(X_test_s)
    metrics_plain = metrics_from_probs_multiclass(probs_plain, y_test)

    clf_he = LogisticRegression(max_iter=5000, solver="lbfgs", class_weight="balanced", multi_class="ovr")
    clf_he.fit(np.vstack([X_train_he, X_val_he]), np.concatenate([y_train, y_val]))
    W_he = clf_he.coef_.astype(float)
    b_he = clf_he.intercept_.astype(float)

    paillier_logits, paillier_time = paillier_inference_multiclass(X_test_he, W_he, b_he, scale=PAILLIER_SCALE, key_size=PAILLIER_KEY)
    probs_paillier = softmax_rows(paillier_logits)
    metrics_paillier = metrics_from_probs_multiclass(probs_paillier, y_test)

    ckks_single_logits, ckks_single_time = ckks_single_inference_multiclass(X_test_he, W_he, b_he)
    probs_ckks_single = softmax_rows(ckks_single_logits)
    metrics_ckks_single = metrics_from_probs_multiclass(probs_ckks_single, y_test)

    ckks_batch_logits, ckks_batch_time = ckks_batched_inference_multiclass(X_test_he, W_he, b_he)
    probs_ckks_batch = softmax_rows(ckks_batch_logits)
    metrics_ckks_batch = metrics_from_probs_multiclass(probs_ckks_batch, y_test)

    print("Fold results:")
    print("Plain:", metrics_plain)
    print("Paillier:", metrics_paillier)
    print("CKKS single:", metrics_ckks_single)
    print("CKKS batched:", metrics_ckks_batch)

    fold_results.append({
        "plain": metrics_plain,
        "paillier": metrics_paillier,
        "ckks_single": metrics_ckks_single,
        "ckks_batch": metrics_ckks_batch,
        "time_paillier": paillier_time,
        "time_ckks_single": ckks_single_time,
        "time_ckks_batch": ckks_batch_time
    })

# ---------- aggregate results ----------
def agg(key):
    arr = [fr[key] for fr in fold_results]
    df = pd.DataFrame(arr)
    mean = df.mean().to_dict(); std = df.std().to_dict()
    return {k: f"{mean[k]:.4f} ± {std[k]:.4f}" for k in mean.keys()}

print("\n=== Aggregated (mean ± std) ===")
print("Plaintext:", agg("plain"))
print("Paillier :", agg("paillier"))
print("CKKS sing:", agg("ckks_single"))
print("CKKS bat :", agg("ckks_batch"))

times = pd.DataFrame([{"paillier":fr["time_paillier"], "ckks_single":fr["time_ckks_single"], "ckks_batch":fr["time_ckks_batch"]} for fr in fold_results])
print("\nAverage runtimes (s):", times.mean().to_dict())


Device: cpu
Cora: nodes=2708, features=1433, classes=7

=== Fold 1/3 ===
Epoch 001 loss: 1.9888
Epoch 010 loss: 2.2766
Epoch 020 loss: 3.0047
Epoch 030 loss: 2.3098
Epoch 040 loss: 2.1604
Epoch 050 loss: 2.1533
Epoch 060 loss: 2.1840
Epoch 070 loss: 2.0480
Epoch 080 loss: 2.1108
Epoch 090 loss: 1.9828
Epoch 100 loss: 1.9464
Epoch 110 loss: 1.9816
Epoch 120 loss: 1.9386


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1256: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. Use OneVsRestClassifier(LogisticRegression(..)) instead. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1256: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. Use OneVsRestClassifier(LogisticRegression(..)) instead. Leave it to its default value to avoid this warning.
  warnings.warn(


Fold results:
Plain: {'accuracy': 0.9114064230343301, 'precision_macro': 0.909400286077114, 'recall_macro': 0.9055498846772954, 'f1_macro': 0.9071567001126496, 'roc_auc_ovr': 0.9908061631926993, 'log_loss': 0.296491006857708}
Paillier: {'accuracy': 0.9058693244739756, 'precision_macro': 0.9025339954399005, 'recall_macro': 0.8969480310613244, 'f1_macro': 0.8994537482866439, 'roc_auc_ovr': 0.9907583983821043, 'log_loss': 0.36711613588984704}
CKKS single: {'accuracy': 0.9058693244739756, 'precision_macro': 0.9025339954399005, 'recall_macro': 0.8969480310613244, 'f1_macro': 0.8994537482866439, 'roc_auc_ovr': 0.9907583983821044, 'log_loss': 0.36710645526991526}
CKKS batched: {'accuracy': 0.9058693244739756, 'precision_macro': 0.9025339954399005, 'recall_macro': 0.8969480310613244, 'f1_macro': 0.8994537482866439, 'roc_auc_ovr': 0.9907583983821044, 'log_loss': 0.3671064554108017}

=== Fold 2/3 ===
Epoch 001 loss: 1.9978
Epoch 010 loss: 3.0479
Epoch 020 loss: 3.0990
Epoch 030 loss: 2.3723
Epoc

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1256: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. Use OneVsRestClassifier(LogisticRegression(..)) instead. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1256: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. Use OneVsRestClassifier(LogisticRegression(..)) instead. Leave it to its default value to avoid this warning.
  warnings.warn(


Fold results:
Plain: {'accuracy': 0.9158361018826136, 'precision_macro': 0.9139748467703338, 'recall_macro': 0.9091814516193885, 'f1_macro': 0.9109950896522517, 'roc_auc_ovr': 0.9891989618336839, 'log_loss': 0.3167595654551244}
Paillier: {'accuracy': 0.9102990033222591, 'precision_macro': 0.9040572881394924, 'recall_macro': 0.8999830791390943, 'f1_macro': 0.9011171081860123, 'roc_auc_ovr': 0.9898496552467384, 'log_loss': 0.415960271564153}
CKKS single: {'accuracy': 0.9102990033222591, 'precision_macro': 0.9040572881394924, 'recall_macro': 0.8999830791390943, 'f1_macro': 0.9011171081860123, 'roc_auc_ovr': 0.9898496552467384, 'log_loss': 0.4159515399429581}
CKKS batched: {'accuracy': 0.9102990033222591, 'precision_macro': 0.9040572881394924, 'recall_macro': 0.8999830791390943, 'f1_macro': 0.9011171081860123, 'roc_auc_ovr': 0.9898496552467384, 'log_loss': 0.4159515403757761}

=== Fold 3/3 ===
Epoch 001 loss: 1.9797
Epoch 010 loss: 2.9254
Epoch 020 loss: 2.1035
Epoch 030 loss: 2.3157
Epoch

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1256: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. Use OneVsRestClassifier(LogisticRegression(..)) instead. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1256: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. Use OneVsRestClassifier(LogisticRegression(..)) instead. Leave it to its default value to avoid this warning.
  warnings.warn(


Fold results:
Plain: {'accuracy': 0.9146341463414634, 'precision_macro': 0.9015731788711149, 'recall_macro': 0.9209579309250868, 'f1_macro': 0.9104020562431134, 'roc_auc_ovr': 0.9889175071626605, 'log_loss': 0.32832052878606677}
Paillier: {'accuracy': 0.9168514412416852, 'precision_macro': 0.9058115845369911, 'recall_macro': 0.9197434780077697, 'f1_macro': 0.9124233138113157, 'roc_auc_ovr': 0.9921910850729109, 'log_loss': 0.3617116302370046}
CKKS single: {'accuracy': 0.9168514412416852, 'precision_macro': 0.9058115845369911, 'recall_macro': 0.9197434780077697, 'f1_macro': 0.9124233138113157, 'roc_auc_ovr': 0.992189288059906, 'log_loss': 0.3617248367130159}
CKKS batched: {'accuracy': 0.9168514412416852, 'precision_macro': 0.9058115845369911, 'recall_macro': 0.9197434780077697, 'f1_macro': 0.9124233138113157, 'roc_auc_ovr': 0.992189288059906, 'log_loss': 0.36172483707146147}

=== Aggregated (mean ± std) ===
Plaintext: {'accuracy': '0.9140 ± 0.0023', 'precision_macro': '0.9083 ± 0.0063', 